# Mass-Preserving Random Forest Downscaling of ODIAC CO₂ Emissions
## From 1/120° (~1 km) to 1/480° (~250 m) — Shanghai Pudong & Metropolitan City of Milan

**Manuscript**
Romanato, N., Wang, M., Maragno, D. (2026). *Earth Observation Predictors of
Sub-Kilometre Anthropogenic CO₂ Emissions are Shaped by City Morphology:
Evidence from Shanghai and Milan.* Submitted to *Environmental Modelling & Software*.

**Archive** — code: `https://github.com/nromanato/odiac-rf-downscaling` · data and notebook: DOI `10.5281/zenodo.22707984`

---

### What this notebook does

1. Sums the 12 monthly ODIAC GeoTIFFs into an annual raster and clips it to the
   **bounding box** of the study-area polygon (paper §4.2).
2. Builds a ten-band Earth Observation predictor stack at 1/480° on Google Earth
   Engine, aligned to the ODIAC parent grid through `crsTransform / 4` (Table 1).
3. Aggregates the fine predictors to the 1/120° training grid using the dual
   block-mean / block-maximum scheme (§4.3, Eq. 1–2).
4. Trains a Random Forest in `log1p` target space under Spatial Block GroupKFold
   cross-validation, with gap-constrained hyperparameter selection (§4.5).
5. Redistributes each parent-cell ODIAC total across its 16 child pixels in
   proportion to the predicted score, conserving mass exactly (§4.4, §4.6).
6. Computes zonal statistics on administrative units, stratifies them by
   population density and per-capita emission outlier status (§4.7).
7. Quantifies redistribution stability by spatial block bootstrap (§4.8).
8. Verifies the run against the values reported in the manuscript.

### How to run it

The notebook is parameterised by a single `CITY` variable. Choose one city,
run every cell from top to bottom, then repeat for the other city. Nothing
else is city-specific: there is no hard-coded city logic anywhere below the
configuration cell.

Input files are **not** read from Google Drive. They are uploaded into the
session at the start of each run by cell 0.4, which lists exactly what is
missing and accepts the files through the Colab upload widget or the Files
panel. Outside Colab, place the same files in a `data/` folder next to this
notebook.

### Input data

| Input | Files | Source |
|---|---|---|
| ODIAC 2023 (release 2024) | `ODIAC_2024_01..12_<Shanghai\|Milano>.tif` | NIES — https://db.cger.nies.go.jp/dataset/ODIAC/ |
| Study-area polygon | `pudong_aoi.*` / `milano_aoi.*` | Zenodo deposit |
| Administrative units | `shanghai_jiedao_adm8.*` / `milano_comuni_adm8.*` | OpenStreetMap, `admin_level=8` |
| EO predictors | retrieved at runtime | Google Earth Engine (Table 1) |

A Google Earth Engine account is required unless a previously exported
predictor stack is supplied, in which case the Earth Engine steps are skipped
(see cell 0.4).

**Runtime** — roughly 25–40 minutes per city on a standard Colab CPU runtime,
of which the bootstrap in Section 9 is the largest share.

**Licence** — code MIT, data products CC-BY-4.0.

## Section 0 — Setup

Installs the Python dependencies, selects the city, gathers the input files for
this session, authenticates Google Earth Engine when required, and loads the
study-area polygon and the administrative units.

In [ ]:
# -----------------------------------------------------------------------------
# 0.1 — Install Python dependencies
# -----------------------------------------------------------------------------
# Run this cell once per Colab session. The runtime may need to be restarted
# after the first installation.
#
# Note: when running locally (outside Colab), use the requirements.txt file
# provided in the repository instead.
# -----------------------------------------------------------------------------

!pip install -q earthengine-api geemap rasterio rasterstats geopandas xgboost \
                scikit-learn matplotlib seaborn pandas numpy scipy folium

In [ ]:
# -----------------------------------------------------------------------------
# 0.2 — Configuration: select the city to process
# -----------------------------------------------------------------------------
# CITY is mandatory and has no default: the notebook is a two-city comparative
# pipeline and every downstream parameter is resolved from CONFIG[CITY].
# Set it to exactly one of "shanghai_pudong" or "milan", then run all cells.
# -----------------------------------------------------------------------------

CITY = None           # <<< REQUIRED: "shanghai_pudong"  or  "milan"

# Google Earth Engine project. Replace with your own authorised project ID.
GEE_PROJECT = "ee-nromanato"

# Permanent archive of the input data and of this notebook.
# Replace the placeholder once the Zenodo DOI has been reserved.
ZENODO_DOI  = "10.5281/zenodo.22707984"
GITHUB_REPO = "https://github.com/nromanato/odiac-rf-downscaling"

VALID_CITIES = ("shanghai_pudong", "milan")
if CITY is None:
    raise ValueError(
        "CITY is not set. Edit this cell and choose one of "
        f"{VALID_CITIES} before running the notebook."
    )
if CITY not in VALID_CITIES:
    raise ValueError(f"CITY must be one of {VALID_CITIES}; got {CITY!r}")

print(f"Selected city: {CITY}")
print(f"Data archive:  https://doi.org/{ZENODO_DOI}")

In [ ]:
# -----------------------------------------------------------------------------
# 0.3 — Imports, data folder and master CONFIG dictionary
# -----------------------------------------------------------------------------
# All parameters live in CONFIG[CITY]. Do not hard-code city-specific values
# anywhere below this cell. The ten predictor bands and three structural
# maxima are identical across both cities by design (paper §4.3): only the
# AOI, file paths, CRS, and the city-specific selected hyperparameters differ.
# -----------------------------------------------------------------------------

import os
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# All input files live in a single session folder, populated by cell 0.4.
DATA_DIR = Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Shared parameters (paper §4.3, §4.4, §4.5)
SHARED = {
    "year": 2023,                       # ODIAC reference year (release 2024)
    "upscale": 4,                       # 1/120° → 1/480° (factor 4 → 16 fine pixels per parent)
    "spatial_block_size": 8,            # 8×8 coarse cells per CV block (~ 8 × 8 km)
    "outlier_cap_percentile": 99.5,     # cap on training target (paper §4.4)
    "n_cv_folds": 5,                    # Spatial Block GroupKFold (paper §4.5)
    "delta_gap_threshold": 0.15,        # train/CV gap constraint (paper §4.5)
    "param_grid": {
        "rf__max_depth":        [3, 4, 5, 6, 7, 8, 10],
        "rf__min_samples_leaf": [10, 15, 20, 30],
        "rf__n_estimators":     [200, 300],
        "rf__max_features":     ["sqrt"],
    },
    "random_state": 42,
}

# Ten predictor bands (paper §4.3, Table 1) and the three structural bands
# for which the within-block maximum is additionally retained.
BASE_BANDS = [
    "built_frac", "built_nres_frac", "pop_count",
    "wc_built", "wc_tree", "wc_crop", "wc_water",
    "no2_trop", "ndvi", "elevation",
]
MAX_BANDS = ["built_frac", "built_nres_frac", "pop_count"]

# Bands for which log1p is applied prior to RF training (paper §4.4)
LOG_COLS  = ["pop_count_mean", "pop_count_max", "no2_trop_mean"]

# City-specific parameters
CONFIG = {
    "shanghai_pudong": {
        **SHARED,
        "city_label": "Shanghai Pudong",
        "aoi_shp": str(DATA_DIR / "pudong_aoi.shp"),
        "zones_shp": str(DATA_DIR / "shanghai_jiedao_adm8.shp"),
        "zones_label_field": "name_en",
        "odiac_pattern": "ODIAC_2024_*_Shanghai.tif",  # 12 monthly files, uploaded in cell 0.4
        "crs_utm": "EPSG:32651",                       # UTM 51N
        "lat_for_pixel_size": 30.6,
        "out_subdir": "shanghai_pudong",
    },
    "milan": {
        **SHARED,
        "city_label": "Metropolitan City of Milan",
        "aoi_shp": str(DATA_DIR / "milano_aoi.shp"),
        "zones_shp": str(DATA_DIR / "milano_comuni_adm8.shp"),
        "zones_label_field": "name",
        "odiac_pattern": "ODIAC_2024_*_Milano.tif",
        "crs_utm": "EPSG:32632",                       # UTM 32N
        "lat_for_pixel_size": 45.5,
        "out_subdir": "milan",
    },
}

cfg = CONFIG[CITY]
RESULT_DIR = Path("results") / cfg["out_subdir"]
RESULT_DIR.mkdir(parents=True, exist_ok=True)

YEAR               = cfg["year"]
UPSCALE            = cfg["upscale"]
SPATIAL_BLOCK_SIZE = cfg["spatial_block_size"]

print(f"City:          {cfg['city_label']}")
print(f"Reference year: {YEAR}")
print(f"Upscale factor: {UPSCALE} (1/120° → 1/480°)")
print(f"Output dir:     {RESULT_DIR}")

In [ ]:
# -----------------------------------------------------------------------------
# 0.4 — Input files for this session
# -----------------------------------------------------------------------------
# Every input is uploaded into the runtime at the start of each session; no
# Google Drive mount is involved. The cell builds the list of files required
# for the selected city, reports what is missing by name, and on Colab opens
# the upload widget until the list is complete. Files dragged into the Files
# panel beforehand are detected and moved into data/ automatically.
#
# Optional: if a predictor stack exported by a previous run is supplied as
#   gee_features_fine_<city>.tif
# the Earth Engine steps (cells 0.5 and Section 2) are skipped entirely, which
# allows the pipeline to be reproduced without an Earth Engine account.
# -----------------------------------------------------------------------------

import shutil

SHAPE_EXT_REQUIRED = [".shp", ".shx", ".dbf", ".prj"]
SHAPE_EXT_OPTIONAL = [".cpg", ".qix", ".qmd"]

odiac_files = [cfg["odiac_pattern"].replace("*", f"{m:02d}") for m in range(1, 13)]
shape_stems = [Path(cfg["aoi_shp"]).stem, Path(cfg["zones_shp"]).stem]
shape_files = [f"{stem}{ext}" for stem in shape_stems for ext in SHAPE_EXT_REQUIRED]

REQUIRED_FILES = odiac_files + shape_files
CACHED_FEATURES = DATA_DIR / f"gee_features_fine_{cfg['out_subdir']}.tif"


def _collect_loose_files():
    """Move any required file already present in the working directory into data/."""
    wanted = set(REQUIRED_FILES) | {CACHED_FEATURES.name}
    wanted |= {f"{stem}{ext}" for stem in shape_stems for ext in SHAPE_EXT_OPTIONAL}
    moved = 0
    for name in wanted:
        loose = Path(name)
        if loose.exists() and not (DATA_DIR / name).exists():
            shutil.move(str(loose), str(DATA_DIR / name))
            moved += 1
    return moved


def _missing_files():
    return [f for f in REQUIRED_FILES if not (DATA_DIR / f).exists()]


try:
    from google.colab import files as _colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

_collect_loose_files()
missing = _missing_files()

if missing and IN_COLAB:
    print(f"{len(missing)} of {len(REQUIRED_FILES)} input files are missing for "
          f"{cfg['city_label']}.\n")
    while missing:
        print("Still missing:")
        for f in missing:
            print(f"   {f}")
        print("\nSelect the missing files in the dialog below "
              "(multiple selection is allowed).\n")
        _colab_files.upload()
        _collect_loose_files()
        new_missing = _missing_files()
        if new_missing == missing:
            raise FileNotFoundError(
                "No new input files were received. Upload the files listed above, "
                "or place them in the data/ folder, then run this cell again."
            )
        missing = new_missing
        print()

if missing:
    raise FileNotFoundError(
        f"Missing {len(missing)} input files in {DATA_DIR.resolve()}:\n  "
        + "\n  ".join(missing)
        + f"\n\nDownload them from https://doi.org/{ZENODO_DOI} and place them in "
          f"{DATA_DIR.resolve()}."
    )

USE_CACHED_FEATURES = CACHED_FEATURES.exists()

print(f"All {len(REQUIRED_FILES)} required input files are present in "
      f"{DATA_DIR.resolve()}")
print(f"  ODIAC monthly rasters:  {len(odiac_files)}")
print(f"  Shapefile components:   {len(shape_files)}")
print(f"  Cached predictor stack: "
      f"{'yes — Earth Engine steps will be skipped' if USE_CACHED_FEATURES else 'no — predictors will be built on Earth Engine'}")

In [ ]:
# -----------------------------------------------------------------------------
# 0.5 — Google Earth Engine authentication
# -----------------------------------------------------------------------------
# Set GEE_PROJECT in cell 0.2 to your authorised Earth Engine project. The first
# run in a new session triggers an interactive authentication flow; later runs
# reuse the cached token. Authentication is skipped when a cached predictor
# stack was supplied in cell 0.4.
# -----------------------------------------------------------------------------

import ee
import geemap

EE_READY = False

if USE_CACHED_FEATURES:
    print("Cached predictor stack detected — Earth Engine is not required.")
    print(f"Remove {CACHED_FEATURES.name} from data/ to rebuild the stack from Earth Engine.")
else:
    try:
        ee.Initialize(project=GEE_PROJECT)
        print("GEE already authenticated and initialised.")
    except Exception:
        ee.Authenticate()
        ee.Initialize(project=GEE_PROJECT)
        print("GEE authenticated and initialised.")
    EE_READY = True

In [ ]:
# -----------------------------------------------------------------------------
# 0.6 — Load AOI polygon and administrative zones (OSM admin_level=8)
# -----------------------------------------------------------------------------
# The AOI shapefile defines the analysis polygon (35 jiedao for Shanghai
# Pudong, 140 comuni for Milan).
# The zones shapefile is used downstream for zonal statistics (paper §4.7).
# Both are uploaded into data/ by cell 0.4.
# -----------------------------------------------------------------------------

import geopandas as gpd
from shapely.ops import unary_union

gdf_aoi  = gpd.read_file(cfg["aoi_shp"]).to_crs("EPSG:4326")
aoi_geom = unary_union(gdf_aoi.geometry)
AOI_BBOX = list(aoi_geom.bounds)
ROI      = (ee.Geometry.Polygon(list(aoi_geom.__geo_interface__["coordinates"][0]))
            if EE_READY else None)

gdf_zones = gpd.read_file(cfg["zones_shp"]).to_crs("EPSG:4326")
gdf_zones["_label"] = gdf_zones[cfg["zones_label_field"]].fillna("n/d")

# Compute zone areas in km² using an equal-area projection appropriate to the
# city. EPSG:4326 (lat/lon) cannot be used directly for area calculation;
# reprojecting to a metric equal-area CRS gives accurate km² values without
# the distortion of UTM at the AOI scale.
#   - Shanghai Pudong → EPSG:3035 is European; use EPSG:6933 (World Equal Area)
#   - Milan           → EPSG:3035 (ETRS89-LAEA Europe, official EEA standard)
equal_area_crs = "EPSG:3035" if cfg["out_subdir"].startswith("milan") else "EPSG:6933"
gdf_zones["_area_km2"] = gdf_zones.to_crs(equal_area_crs).area / 1e6

print(f"AOI bounds:    {[round(x, 4) for x in AOI_BBOX]}")
print(f"Admin zones:   {len(gdf_zones)} units (OSM admin_level=8)")
print(f"\nZone area statistics (km², via {equal_area_crs}):")
print(f"  Total area:  {gdf_zones['_area_km2'].sum():,.1f} km²")
print(f"  Mean:        {gdf_zones['_area_km2'].mean():,.2f} km²")
print(f"  Median:      {gdf_zones['_area_km2'].median():,.2f} km²")
print(f"  Min / Max:   {gdf_zones['_area_km2'].min():,.2f} / {gdf_zones['_area_km2'].max():,.2f} km²")
print(f"\nSmallest 3 zones:")
for _, r in gdf_zones.nsmallest(3, "_area_km2")[["_label", "_area_km2"]].iterrows():
    print(f"  {r['_label']:<40} {r['_area_km2']:>8.2f} km²")
print(f"\nLargest 3 zones:")
for _, r in gdf_zones.nlargest(3, "_area_km2")[["_label", "_area_km2"]].iterrows():
    print(f"  {r['_label']:<40} {r['_area_km2']:>8.2f} km²")

## Section 1 — ODIAC Inventory Preprocessing

Sums the twelve monthly ODIAC GeoTIFFs into a single annual raster in
tCO₂ cell⁻¹ yr⁻¹ and clips it to the **bounding box** of the study-area
polygon (paper §4.2). The polygon itself is deliberately not used as the clip
geometry at this stage: border cells that the polygon would truncate carry
valid predictor values, and the polygon mask is applied only to the downscaled
output in Section 6. The clipped extent defines the reference grid for every
Earth Engine export that follows.

In [ ]:
import rasterio
from rasterio.mask import mask

_t0 = time.time()

ODIAC_DIR  = DATA_DIR
tif_files  = sorted(ODIAC_DIR.glob(cfg["odiac_pattern"]))

if len(tif_files) != 12:
    raise FileNotFoundError(
        f"Expected 12 monthly ODIAC files matching {cfg['odiac_pattern']!r}, "
        f"found {len(tif_files)} in {ODIAC_DIR}. "
        f"Re-run cell 0.4 to upload the missing files."
    )

print(f"Loading and clipping {len(tif_files)} monthly ODIAC files to AOI...")

from shapely.geometry import box
clip_geom = [box(*aoi_geom.bounds).__geo_interface__]

monthly_arrays   = []
fine_transform   = None
odiac_profile    = None

for f in tif_files:
    with rasterio.open(f) as src:
        clipped, clip_transform = mask(src, clip_geom, crop=True)
        arr = clipped[0].astype(np.float32)
        if fine_transform is None:
            fine_transform = clip_transform
            odiac_profile  = src.profile.copy()
            odiac_profile.update({
                "height":    arr.shape[0],
                "width":     arr.shape[1],
                "transform": clip_transform,
            })
    monthly_arrays.append(arr)

# Sum 12 months → annual total, clip negatives to zero (numerical noise)
annual_tco2_1km = np.stack(monthly_arrays, axis=0).sum(axis=0)
annual_tco2_1km = np.clip(
    np.nan_to_num(annual_tco2_1km, nan=0.0), 0, None
).astype(np.float32)

h_c, w_c = annual_tco2_1km.shape

AOI_BBOX = [
    fine_transform.c,
    fine_transform.f + fine_transform.e * h_c,
    fine_transform.c + fine_transform.a * w_c,
    fine_transform.f,
]
ROI = ee.Geometry.BBox(*AOI_BBOX) if EE_READY else None

# Pixel size in metres for diagnostics
px_lon_m = fine_transform.a * 111320 * np.cos(np.radians(cfg["lat_for_pixel_size"]))
px_lat_m = abs(fine_transform.e) * 111320

# --- Save annual ODIAC raster (1/120°, bbox-clipped, tCO₂/cell/yr) ---

odiac_profile_out = odiac_profile.copy()
odiac_profile_out.update({
    "dtype":    "float32",
    "count":    1,
    "nodata":   0.0,
    "compress": "lzw",
})

odiac_annual_tif = RESULT_DIR / f"odiac_annual_{cfg['out_subdir']}_1-120deg.tif"
with rasterio.open(odiac_annual_tif, "w", **odiac_profile_out) as dst:
    dst.write(annual_tco2_1km, 1)
    dst.set_band_description(1, f"ODIAC {cfg['year']} annual emissions (tCO2/cell/yr)")

print(f"\nSaved annual ODIAC: {odiac_annual_tif}")

print(f"\nAnnual ODIAC raster — {cfg['city_label']}:")
print(f"  Shape:            {annual_tco2_1km.shape}")
print(f"  Pixel size:       {fine_transform.a:.4f}° ≈ {px_lon_m:.0f} m × {px_lat_m:.0f} m")
print(f"  Total emissions:  {annual_tco2_1km.sum():,.0f} tCO₂/yr")
print(f"  Max cell:         {annual_tco2_1km.max():,.0f} tCO₂  (largest point source)")
print(f"  Non-zero cells:   {(annual_tco2_1km > 0).sum()} / {annual_tco2_1km.size}")
print(f"\nElapsed: {time.time() - _t0:.1f} s")

## Section 2 — Earth Observation Predictor Extraction

Extracts the ten predictor bands listed in paper Table 1 from Google Earth
Engine, aligned to the ODIAC parent grid by dividing the parent affine
transform by 4 (paper §4.3). This guarantees exact 4×4 nesting between
1/120° and 1/480° pixels.

**Note on resampling:** `resample('bilinear')` is used rather than
`reduceResolution` because the latter exceeds GEE compute limits when the
source band (e.g. GHSL at 10 m) has many input pixels per output cell.

**VIIRS nighttime lights are explicitly excluded** (paper §4.1, §4.3) to
avoid circularity with the NTL-based spatial disaggregation underlying
ODIAC construction (Oda and Maksyutov, 2011).

In [ ]:
_t0 = time.time()

EXPORT_CRS = "EPSG:4326"

# 1/480° exact transform: divide ODIAC parent pixel size by UPSCALE
EXPORT_TRANSFORM_FINE = [
    fine_transform.a / UPSCALE,   # lon pixel size
    0,
    fine_transform.c,             # lon origin (NW corner)
    0,
    fine_transform.e / UPSCALE,   # lat pixel size (negative)
    fine_transform.f,             # lat origin (NW corner)
]


def ghsl_epoch_for_year(y: int) -> str:
    """Return the GHSL P2023A epoch string ('2020' or '2025') for a given year."""
    return "2020" if y <= 2020 else "2025"


def to_grid(img: ee.Image) -> ee.Image:
    """
    Resample a GEE image to the 1/480° target grid via bilinear interpolation,
    aligned to the ODIAC parent grid by crsTransform (not scale).

    Using crsTransform instead of scale ensures pixel-perfect alignment with
    the parent ODIAC grid (4×4 fine pixels per parent cell exactly).
    """
    return (img
            .resample("bilinear")
            .reproject(crs=EXPORT_CRS, crsTransform=EXPORT_TRANSFORM_FINE))

feature_tif = RESULT_DIR / "gee_features_fine.tif"

if USE_CACHED_FEATURES:
    # A predictor stack exported by an earlier run was supplied in cell 0.4:
    # the Earth Engine build below is skipped and the stack is used as is.
    shutil.copy(CACHED_FEATURES, feature_tif)
    print(f"Using cached predictor stack: {CACHED_FEATURES.name}")
    print("Earth Engine export skipped.")
else:
    feature_images = []
    print("Building Earth Observation predictor stack on GEE...")

    # 1. GHSL built-up surface fraction and non-residential built-up fraction
    #    Native 10 m → 1/480°. Scaled by 1e4 (GHSL stores fractions × 10000).
    ghsl_built = ee.Image(f"JRC/GHSL/P2023A/GHS_BUILT_S/{ghsl_epoch_for_year(YEAR)}")
    feature_images.append(to_grid(ghsl_built.select("built_surface").divide(10000.0)).rename("built_frac"))
    feature_images.append(to_grid(ghsl_built.select("built_surface_nres").divide(10000.0)).rename("built_nres_frac"))
    print("  built_frac, built_nres_frac (GHSL P2023A)")

    # 2. GHSL population count
    #    Native 100 m → 1/480°.
    ghsl_pop = ee.Image(f"JRC/GHSL/P2023A/GHS_POP/{ghsl_epoch_for_year(YEAR)}")
    feature_images.append(to_grid(ghsl_pop.select("population_count")).rename("pop_count"))
    print("  pop_count (GHSL P2023A)")

    # 3. ESA WorldCover v200 fractions for four classes
    #    Native 10 m → 1/480°. Each band is the fraction of fine pixel covered by that class.
    wc = ee.ImageCollection("ESA/WorldCover/v200").first().select("Map")
    feature_images.extend([
        to_grid(wc.eq(50)).rename("wc_built"),   # built-up
        to_grid(wc.eq(10)).rename("wc_tree"),    # tree cover
        to_grid(wc.eq(40)).rename("wc_crop"),    # cropland
        to_grid(wc.eq(80)).rename("wc_water"),   # permanent water bodies
    ])
    print("  wc_built, wc_tree, wc_crop, wc_water (ESA WorldCover v200)")

    # 4. Sentinel-5P TROPOMI tropospheric NO₂ column density
    #    Annual mean composite. Native ~3.5 × 5.5 km (post-Aug-2019 footprint).
    no2_ic = (ee.ImageCollection("COPERNICUS/S5P/OFFL/L3_NO2")
              .filterDate(f"{YEAR}-01-01", f"{YEAR + 1}-01-01")
              .filterBounds(ROI))
    no2 = (no2_ic
           .select("tropospheric_NO2_column_number_density")
           .mean()
           .setDefaultProjection(no2_ic.first().select(0).projection()))
    feature_images.append(to_grid(no2).rename("no2_trop"))
    print("  no2_trop (Sentinel-5P OFFL L3)")

    # 5. Sentinel-2 NDVI annual median composite (scene cloud cover ≤ 20%)
    s2_ic = (ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
             .filterDate(f"{YEAR}-01-01", f"{YEAR + 1}-01-01")
             .filterBounds(ROI)
             .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", 20)))
    ndvi = (s2_ic
            .map(lambda img: img.normalizedDifference(["B8", "B4"]).rename("ndvi"))
            .median()
            .setDefaultProjection(s2_ic.first().select("B8").projection()))
    feature_images.append(to_grid(ndvi).rename("ndvi"))
    print("  ndvi (Sentinel-2 SR Harmonized, annual median)")

    # 6. NASADEM elevation (30 m → 1/480°)
    dem = ee.Image("NASA/NASADEM_HGT/001").select("elevation")
    feature_images.append(to_grid(dem).rename("elevation"))
    print("  elevation (NASADEM)")

    # Assemble multi-band image and export
    feature_img = ee.Image.cat(feature_images).toFloat().clip(ROI)
    feature_tif = RESULT_DIR / "gee_features_fine.tif"

    #     Export: use Export.image.toDrive for AOIs that exceed the
    #     50 MB direct-download limit (typical for Milan due to large area
    #     × Sentinel-2 10m native resolution). The export is asynchronous
    #     and we poll until completion.
    print(f"\nExporting predictor stack at 1/480°...")
    print(f"Expected shape: (10 bands, {h_c * UPSCALE}, {w_c * UPSCALE})")

    # Try direct download first (works for compact AOIs like Shanghai Pudong)
    direct_download_ok = False
    try:
        geemap.ee_export_image(
            feature_img,
            filename=str(feature_tif),
            crs=EXPORT_CRS,
            crs_transform=EXPORT_TRANSFORM_FINE,
            region=ROI,
            file_per_band=False,
        )
        if feature_tif.exists() and feature_tif.stat().st_size > 0:
            direct_download_ok = True
            print("  Direct download succeeded.")
    except Exception as e:
        print(f"  Direct download failed ({type(e).__name__}). Falling back to Drive export.")

    # Fallback: batch export to Google Drive (no size limit)
    if not direct_download_ok:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        drive_export_folder = "ODIAC_downscaling_temp"
        drive_filename      = f"gee_features_fine_{cfg['out_subdir']}"

        task = ee.batch.Export.image.toDrive(
            image=feature_img,
            description=drive_filename,
            folder=drive_export_folder,
            fileNamePrefix=drive_filename,
            crs=EXPORT_CRS,
            crsTransform=EXPORT_TRANSFORM_FINE,
            region=ROI,
            maxPixels=1e10,
            fileFormat="GeoTIFF",
        )
        task.start()
        print(f"  Drive export started (task id: {task.id}).")
        print(f"  Polling status (this typically takes 2–5 minutes for Milan)...")

        import time as _time
        poll_interval = 15  # seconds
        max_wait      = 900  # 15 minutes
        elapsed_poll  = 0
        while elapsed_poll < max_wait:
            status = task.status()
            state  = status.get("state", "UNKNOWN")
            print(f"    [t+{elapsed_poll:>3d}s] state = {state}")
            if state in ("COMPLETED", "FAILED", "CANCELLED"):
                break
            _time.sleep(poll_interval)
            elapsed_poll += poll_interval

        if state != "COMPLETED":
            raise RuntimeError(
                f"GEE Drive export did not complete (final state: {state}). "
                f"Error message: {status.get('error_message', '<none>')}"
            )

        # Locate the exported file on Drive and copy it to the expected path
        drive_path = Path(f"/content/drive/MyDrive/{drive_export_folder}/{drive_filename}.tif")
        if not drive_path.exists():
            raise FileNotFoundError(
                f"Drive export completed but file not found at {drive_path}. "
                f"Check Drive folder '{drive_export_folder}' manually."
            )
        import shutil
        shutil.copy(drive_path, feature_tif)
        print(f"  Drive export complete, copied to {feature_tif}.")

# Load back from disk
with rasterio.open(feature_tif) as src:
    features_fine = src.read()
    profile_fine  = src.profile

# Mask invalid pixels (large negatives, NaN, Inf) to zero
features_fine = np.where(
    np.isfinite(features_fine) & (features_fine > -100),
    features_fine, 0.0,
).astype(np.float32)

# Reconcile GEE-returned shape with the expected fine grid shape.
expected_shape = (len(BASE_BANDS), h_c * UPSCALE, w_c * UPSCALE)
if features_fine.shape != expected_shape:
    print(f"  Note: GEE returned {features_fine.shape}, expected {expected_shape}.")
    print(f"  Padding to expected shape (padded pixels are outside the AOI).")

    pad_h = expected_shape[1] - features_fine.shape[1]
    pad_w = expected_shape[2] - features_fine.shape[2]

    if pad_h < 0 or pad_w < 0:
        # GEE returned MORE pixels than expected (rare; border rounding):
        # crop to expected shape instead.
        features_fine = features_fine[:, :expected_shape[1], :expected_shape[2]]
    else:
        features_fine = np.pad(
            features_fine,
            pad_width=((0, 0), (0, pad_h), (0, pad_w)),
            mode="constant",
            constant_values=0.0,
        )

assert features_fine.shape == expected_shape, (
    f"Could not reconcile GEE export shape: got {features_fine.shape}, "
    f"expected {expected_shape}"
)

# Validate each band has signal
for i, name in enumerate(BASE_BANDS):
    assert np.isfinite(features_fine[i]).any(), f"Band {name} is all NaN/Inf"
    assert features_fine[i].std() > 0, f"Band {name} is constant"

print(f"\nPredictor stack ready: shape {features_fine.shape}")

print(f"Elapsed: {time.time() - _t0:.1f} s")

## Section 3 — Training Dataset Construction

Aggregates the 1/480° predictor stack to the 1/120° training grid using
the dual aggregation scheme defined in paper §4.3, Eq. (1)–(2): block
mean for all 10 bands, plus block maximum for the three structural bands
(built_frac, built_nres_frac, pop_count). This produces a 13-dimensional
feature vector per coarse cell.

Training at the coarse 1/120° resolution avoids pseudo-replication
(paper §4.4): the model is evaluated at the spatial scale at which it
was trained.

In [ ]:
_t0 = time.time()


def preprocess_features(df: pd.DataFrame) -> np.ndarray:
    """
    Apply log1p to right-skewed predictor columns (population, NO₂) and
    convert to a float32 array with NaN/Inf masked to zero.

    Must be called identically on training data (coarse grid) and prediction
    data (fine grid) to ensure no train/predict mismatch (paper §4.4).
    """
    df_out = df.copy()
    for c in LOG_COLS:
        if c in df_out.columns:
            df_out[c] = np.log1p(df_out[c].clip(lower=0))
    return np.nan_to_num(
        df_out.values.astype(np.float32),
        nan=0.0, posinf=0.0, neginf=0.0,
    )


print("Aggregating 1/480° predictors to 1/120° training grid (dual aggregation)...")

coarse_data = {}
for i, name in enumerate(BASE_BANDS):
    b_fine   = features_fine[i]                                      # (h_c*4, w_c*4)
    b_blocks = b_fine.reshape(h_c, UPSCALE, w_c, UPSCALE)             # (h_c, 4, w_c, 4)

    # Block mean (always)
    coarse_data[f"{name}_mean"] = b_blocks.mean(axis=(1, 3)).flatten()

    # Block maximum (structural bands only — paper §4.3)
    if name in MAX_BANDS:
        coarse_data[f"{name}_max"] = b_blocks.max(axis=(1, 3)).flatten()

df_coarse      = pd.DataFrame(coarse_data)
FINAL_FEATURES = list(df_coarse.columns)

# Target: annual CO₂ per coarse cell, clipped to non-negative
y_raw = np.clip(
    np.nan_to_num(annual_tco2_1km.flatten().astype(np.float32), nan=0.0), 0, None
)
valid = np.isfinite(y_raw) & (y_raw >= 0)

X_all = preprocess_features(df_coarse[valid])
y_all = y_raw[valid]

# Apply p99.5 cap (paper §4.4)
cap_value = np.percentile(y_all, cfg["outlier_cap_percentile"])
y_valid   = np.clip(y_all, 0, cap_value)
X_valid   = X_all

# Spatial block IDs for GroupKFold (paper §4.5)
# Each coarse cell is assigned to an 8×8 block based on its grid position;
# all 64 cells in a block share the same fold, eliminating within-block
# leakage between training and test partitions.
rows_idx, cols_idx = np.indices((h_c, w_c))
groups_valid = (
    (rows_idx // SPATIAL_BLOCK_SIZE) * 10000
    + cols_idx // SPATIAL_BLOCK_SIZE
).flatten()[valid]

print(f"  Predictor vector: {len(FINAL_FEATURES)} dimensions")
print(f"  Features: {FINAL_FEATURES}")
print(f"  Valid samples: {valid.sum()} / {len(y_raw)}")
print(f"  Outlier cap (p{cfg['outlier_cap_percentile']}): {cap_value:,.0f} tCO₂")
print(f"  Capped pixels:  {(y_all > cap_value).sum()}")
print(f"  Spatial CV blocks: {len(np.unique(groups_valid))}")
print(f"\nElapsed: {time.time() - _t0:.1f} s")

## Section 4 — Random Forest Training with Gap-Constrained Selection

Two-stage hyperparameter selection (paper §4.5):
1. **GridSearchCV** over the 7 × 4 × 2 = 56-configuration parameter grid
   using Spatial Block GroupKFold (5 folds, 8×8 km blocks). Selection
   criterion is CV R² in log1p target space.
2. **Gap constraint**: the configuration with the highest CV R² satisfying
   Δ = R²_train − R²_CV ≤ 0.15 is retained. Configurations with Δ > 0.15
   are discarded regardless of CV R² (paper §4.5, Eq. 5).

For Shanghai Pudong the first-pass optimum already satisfies the constraint.
For Milan, the unconstrained optimum (max_depth=10, leaf=10) yields Δ ≈ 0.17
and is discarded; the selected model is (max_depth=10, leaf=15, n_est=200).

Two baselines are also reported: uniform mean and built-fraction-only RF.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.metrics import r2_score
from sklearn.pipeline import Pipeline

_t0 = time.time()


def rf_predict_original(model, X: np.ndarray) -> np.ndarray:
    """Predict in log1p space and back-transform to original CO₂ space."""
    return np.expm1(model.predict(X)).clip(min=0).astype(np.float32)


n_splits = min(cfg["n_cv_folds"], len(np.unique(groups_valid)))
gkf      = GroupKFold(n_splits=n_splits)
print(f"Spatial Block GroupKFold: {n_splits} folds")

# --- Baseline 1: predict the train-fold mean ---
cv_r2_b1 = []
for tr, te in gkf.split(X_valid, y_valid, groups=groups_valid):
    cv_r2_b1.append(r2_score(y_valid[te], np.full(len(te), y_valid[tr].mean())))
print(f"Baseline (uniform mean):       CV R² = {np.mean(cv_r2_b1):+.3f} ± {np.std(cv_r2_b1):.3f}")

# --- Baseline 2: RF with built_frac only ---
idx_built  = FINAL_FEATURES.index("built_frac_mean")
cv_r2_b2   = []
for tr, te in gkf.split(X_valid, y_valid, groups=groups_valid):
    m_b2 = RandomForestRegressor(n_estimators=50, max_depth=5,
                                  random_state=SHARED["random_state"])
    m_b2.fit(X_valid[tr][:, idx_built].reshape(-1, 1), np.log1p(y_valid[tr]))
    cv_r2_b2.append(r2_score(
        y_valid[te],
        rf_predict_original(m_b2, X_valid[te][:, idx_built].reshape(-1, 1)),
    ))
print(f"Baseline (built_frac only):    CV R² = {np.mean(cv_r2_b2):+.3f} ± {np.std(cv_r2_b2):.3f}")

# --- Stage 1: GridSearchCV over the 56-configuration grid ---
print(f"\nStage 1 — GridSearchCV ({7 * 4 * 2} configurations, log1p target space)")

pipeline = Pipeline([("rf", RandomForestRegressor(
    random_state=SHARED["random_state"], n_jobs=-1
))])

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=SHARED["param_grid"],
    cv=gkf,
    scoring="r2",
    n_jobs=-1,
    verbose=0,
    return_train_score=True,
)

y_log = np.log1p(y_valid)
grid_search.fit(X_valid, y_log, groups=groups_valid)

# --- Stage 2: gap-constrained selection (paper §4.5, Eq. 5) ---
results_df = pd.DataFrame(grid_search.cv_results_)
results_df["gap"]               = results_df["mean_train_score"] - results_df["mean_test_score"]
results_df["max_depth"]         = results_df["param_rf__max_depth"]
results_df["min_samples_leaf"]  = results_df["param_rf__min_samples_leaf"]
results_df["n_estimators"]      = results_df["param_rf__n_estimators"]

cols_display = ["max_depth", "min_samples_leaf", "n_estimators",
                "mean_test_score", "mean_train_score", "gap"]
print("\nTop 10 configurations by CV R² (log1p space):")
print(results_df[cols_display].sort_values("mean_test_score", ascending=False)
      .head(10).to_string(index=False))

delta_threshold = SHARED["delta_gap_threshold"]
constrained = results_df[results_df["gap"] < delta_threshold] \
                .sort_values("mean_test_score", ascending=False)

if len(constrained) == 0:
    raise RuntimeError(
        f"No configuration satisfies the gap constraint Δ < {delta_threshold}. "
        f"Inspect grid_search.cv_results_ before proceeding."
    )

best = constrained.iloc[0]
best_params = {
    "max_depth":        int(best["max_depth"]),
    "min_samples_leaf": int(best["min_samples_leaf"]),
    "n_estimators":     int(best["n_estimators"]),
    "max_features":     "sqrt",
}

print(f"\nStage 2 — Selected configuration (highest CV R² with Δ ≤ {delta_threshold}):")
print(f"  Parameters:        {best_params}")
print(f"  CV R²  (log1p):    {best['mean_test_score']:.3f}")
print(f"  Train R² (log1p):  {best['mean_train_score']:.3f}")
print(f"  Gap    (log1p):    {best['gap']:.3f}")

# --- Refit final model on full training set with selected parameters ---
final_model = RandomForestRegressor(
    **best_params, random_state=SHARED["random_state"], n_jobs=-1
)
final_model.fit(X_valid, y_log)

# CV metrics in original CO₂ space (for paper reporting — paper §5.1)
cv_r2_original = []
for tr, te in gkf.split(X_valid, y_valid, groups=groups_valid):
    m = RandomForestRegressor(**best_params,
                               random_state=SHARED["random_state"], n_jobs=-1)
    m.fit(X_valid[tr], np.log1p(y_valid[tr]))
    cv_r2_original.append(r2_score(y_valid[te], rf_predict_original(m, X_valid[te])))

cv_r2_final   = float(np.mean(cv_r2_original))
r2_train_full = r2_score(y_valid, rf_predict_original(final_model, X_valid))
gap_original  = r2_train_full - cv_r2_final

print(f"\nFinal model metrics (original emission space — report in paper):")
print(f"  CV R² (Spatial Block GroupKFold): {cv_r2_final:.3f}")
print(f"  Train R²:                         {r2_train_full:.3f}")
print(f"  Train/CV gap:                     {gap_original:.3f}")
print(f"\nElapsed: {time.time() - _t0:.1f} s")

In [ ]:
# -----------------------------------------------------------------------------
# 4.1 — Sensitivity of CV R² to the outlier cap percentile (paper §5.1, Table 4)
# -----------------------------------------------------------------------------

print("Sensitivity of CV R² to outlier cap percentile (final RF configuration):\n")
print(f"{'Cap percentile':<20} {'CV R² (original)':>20}")
print("-" * 42)
for pct in [99.0, 99.5, 99.9, 100.0]:
    y_cap   = np.clip(y_all, 0, np.percentile(y_all, pct))
    cv_caps = []
    for tr, te in gkf.split(X_all, y_cap, groups=groups_valid):
        m = RandomForestRegressor(**best_params,
                                   random_state=SHARED["random_state"], n_jobs=-1)
        m.fit(X_all[tr], np.log1p(y_cap[tr]))
        cv_caps.append(r2_score(y_cap[te], np.expm1(m.predict(X_all[te])).clip(0)))
    label = f"p{pct}" if pct < 100 else "no cap"
    flag  = "  ← selected" if pct == cfg["outlier_cap_percentile"] else ""
    print(f"{label:<20} {np.mean(cv_caps):>20.3f}{flag}")

## Section 5 — Feature Importance: MDI and Permutation

Both Mean Decrease Impurity and permutation importance (paper §5.2) are
computed. MDI is known to be biased under correlated predictors (Strobl
et al., 2008); permutation importance with n=20 repeats is reported as
the more conservative robustness check. The paper finds that the dominant
predictor is preserved across both metrics — NO₂ for Milan, elevation for
Shanghai Pudong — but secondary rankings shift, consistent with the joint
informational role of correlated urban-density proxies.

In [ ]:
from sklearn.inspection import permutation_importance

# --- 5.1 MDI (Mean Decrease Impurity) ---
fi_mdi = pd.Series(final_model.feature_importances_, index=FINAL_FEATURES)
fi_mdi_sorted = fi_mdi.sort_values(ascending=False)

print(f"Feature importance (MDI) — {cfg['city_label']}:\n")
uniform_ref = 1 / len(FINAL_FEATURES)
print(f"{'Feature':<25} {'MDI':>10} {'× uniform':>12}")
print("-" * 50)
for feat, val in fi_mdi_sorted.items():
    flag = " ★" if val > uniform_ref else ""
    print(f"{feat:<25} {val:>10.4f} {val / uniform_ref:>11.2f}{flag}")
print(f"\n  Uniform reference (1/{len(FINAL_FEATURES)}): {uniform_ref:.4f}")

# Save CSV for downstream comparison
fi_mdi.to_frame("mdi").to_csv(RESULT_DIR / "feature_importance_mdi.csv")

# Bar plot
fig, ax = plt.subplots(figsize=(8, max(4, len(FINAL_FEATURES) * 0.35)))
fi_mdi.sort_values().plot(kind="barh", ax=ax, color="steelblue")
ax.axvline(uniform_ref, color="red", linestyle="--", lw=1,
           label=f"Uniform = {uniform_ref:.3f}")
ax.set_title(f"RF Feature Importance (MDI) — {cfg['city_label']}, {YEAR}")
ax.set_xlabel("Mean Decrease Impurity")
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(RESULT_DIR / "feature_importance_mdi.png", dpi=150, bbox_inches="tight")
plt.show()

# --- 5.2 Permutation importance (n=20 repeats, in log1p target space) ---
print(f"\nComputing permutation importance (n=20 repeats)...")
perm = permutation_importance(
    final_model, X_valid, np.log1p(y_valid),
    n_repeats=20,
    random_state=SHARED["random_state"],
    n_jobs=-1,
)
fi_perm     = pd.Series(perm.importances_mean, index=FINAL_FEATURES)
fi_perm_std = pd.Series(perm.importances_std,  index=FINAL_FEATURES)

fi_perm.to_frame("perm_mean").assign(
    perm_std=fi_perm_std
).to_csv(RESULT_DIR / "feature_importance_permutation.csv")

print(f"\nPermutation importance (mean ± σ, log1p target space):\n")
print(f"{'Feature':<25} {'mean':>10} {'σ':>10}")
print("-" * 50)
for feat, val in fi_perm.sort_values(ascending=False).items():
    print(f"{feat:<25} {val:>10.4f} {fi_perm_std[feat]:>10.4f}")

# Bar plot
fig, ax = plt.subplots(figsize=(8, max(4, len(FINAL_FEATURES) * 0.35)))
fi_perm_sorted = fi_perm.sort_values()
ax.barh(fi_perm_sorted.index, fi_perm_sorted.values,
        xerr=fi_perm_std[fi_perm_sorted.index].values,
        color="steelblue", alpha=0.8)
ax.axvline(0, color="black", lw=0.8)
ax.set_title(f"Permutation Importance (±1σ, n=20) — {cfg['city_label']}, {YEAR}")
ax.set_xlabel("Mean accuracy decrease (log1p target space)")
plt.tight_layout()
plt.savefig(RESULT_DIR / "feature_importance_permutation.png",
            dpi=150, bbox_inches="tight")
plt.show()

## Section 6 — Spatial Prediction and Mass-Preserving Redistribution

Applies the trained RF model to the 1/480° feature grid to obtain a spatial
redistribution score $s_p$ per fine pixel (paper §4.4, Eq. 4). For each
parent ODIAC cell with total emission $E_c$, the score is normalised so
that the 16 fine pixels sum to $E_c$ exactly (paper §4.6, Eq. 6):

$$e_{i,j} = \frac{s_{i,j}}{\sum_{k,l \in P_c} s_{k,l}} \cdot E_c$$

In the degenerate case where all 16 RF scores are zero (e.g. ocean pixels
with no built-up or population signal), emissions are distributed uniformly:
$e_{i,j} = E_c / 16$.

Mass conservation is verified empirically as the relative residual between
input ODIAC total and output downscaled total — typically bounded by
float32 arithmetic precision (< 0.001%, paper §5.1, Table 3).

In [ ]:
from rasterio.features import geometry_mask
from rasterio.transform import Affine, from_bounds

_t0 = time.time()


def rf_predict_fine(model, X: np.ndarray) -> np.ndarray:
    """Predict in log1p space, back-transform, and clip to non-negative."""
    return np.expm1(model.predict(X)).clip(min=0).astype(np.float32)


# --- 6.1 Build fine-grid feature matrix using the same FINAL_FEATURES order ---
print("Building 1/480° prediction feature matrix...")
fine_data = {}
for i, name in enumerate(BASE_BANDS):
    b = features_fine[i]
    fine_data[f"{name}_mean"] = b.flatten()
    if name in MAX_BANDS:
        # On the fine grid the "max" feature is just the pixel value itself
        # (each fine pixel is a single observation, not a block).
        fine_data[f"{name}_max"] = b.flatten()

df_fine = pd.DataFrame(fine_data)[FINAL_FEATURES]
X_fine  = preprocess_features(df_fine)

# --- 6.2 Predict at fine resolution ---
print("Predicting at 1/480°...")
score_fine = rf_predict_fine(final_model, X_fine).reshape(h_c * UPSCALE, w_c * UPSCALE)
print(f"  RF score range: {score_fine.min():.1f} – {score_fine.max():.1f}")

# --- 6.3 Mass-preserving redistribution (paper §4.6, Eq. 6) ---
print("Applying mass-preserving redistribution...")

# Sum of RF scores per parent cell (1/120° grid)
score_sum_coarse = (score_fine
                    .reshape(h_c, UPSCALE, w_c, UPSCALE)
                    .sum(axis=(1, 3)))                                 # (h_c, w_c)

# Broadcast the per-parent sum back to the fine grid
score_sum_up = np.repeat(np.repeat(score_sum_coarse, UPSCALE, axis=0),
                          UPSCALE, axis=1)                              # (h_c*4, w_c*4)

# Broadcast parent ODIAC values to the fine grid
odiac_up = np.repeat(np.repeat(annual_tco2_1km, UPSCALE, axis=0),
                      UPSCALE, axis=1)

# Compute the per-pixel fraction, with uniform fallback for degenerate cells
with np.errstate(divide="ignore", invalid="ignore"):
    fraction = np.where(
        score_sum_up > 0,
        score_fine / score_sum_up,
        1.0 / (UPSCALE * UPSCALE),
    )

downscaled = (fraction * odiac_up).astype(np.float32)
downscaled = np.where(
    np.isfinite(downscaled) & (downscaled >= 0),
    downscaled, 0.0,
).astype(np.float32)

# --- 6.4 Mass conservation verification (paper §4.6, Eq. 7) ---
mass_input    = float(annual_tco2_1km.sum())
mass_output   = float(downscaled.sum())
delta_mass    = abs(mass_output - mass_input)
delta_mass_pct = delta_mass / mass_input * 100

print(f"\nMass conservation:")
print(f"  ODIAC input total:   {mass_input:>15,.0f} tCO₂/yr")
print(f"  Downscaled total:    {mass_output:>15,.0f} tCO₂/yr")
print(f"  Absolute difference: {delta_mass:>15,.0f} tCO₂")
print(f"  Relative difference: {delta_mass_pct:>15.6f}%  (target < 0.001%)")

if delta_mass_pct > 0.001:
    print("  WARNING: mass conservation residual exceeds threshold.")
else:
    print("  Mass conservation verified.")

# --- 6.5 Mask pixels outside the AOI polygon ---
print("\nMasking pixels outside the AOI polygon...")
aoi_transform_fine = from_bounds(
    AOI_BBOX[0], AOI_BBOX[1], AOI_BBOX[2], AOI_BBOX[3],
    w_c * UPSCALE, h_c * UPSCALE,
)
aoi_mask = geometry_mask(
    [aoi_geom.__geo_interface__],
    transform=aoi_transform_fine,
    invert=True,
    out_shape=(h_c * UPSCALE, w_c * UPSCALE),
)
downscaled = np.where(aoi_mask, downscaled, 0.0).astype(np.float32)
print(f"  Active pixels: {aoi_mask.sum()} / {aoi_mask.size} "
      f"({aoi_mask.sum() / aoi_mask.size * 100:.1f}%)")

# --- 6.6 Save downscaled GeoTIFF with clean axis-aligned transform ---
# We rebuild the affine from scratch (rather than copying profile_fine) to
# guarantee no rotation/shear artefacts from intermediate GEE reprojections.
transform_fine_clean = Affine(
    fine_transform.a / UPSCALE, 0.0, fine_transform.c,
    0.0, fine_transform.e / UPSCALE, fine_transform.f,
)
profile_out = {
    "driver":    "GTiff",
    "dtype":     "float32",
    "width":     w_c * UPSCALE,
    "height":    h_c * UPSCALE,
    "count":     1,
    "crs":       "EPSG:4326",
    "transform": transform_fine_clean,
    "nodata":    0.0,
    "compress":  "lzw",
}

out_tif = RESULT_DIR / f"downscaled_CO2_{cfg['out_subdir']}_1-480deg.tif"
with rasterio.open(out_tif, "w", **profile_out) as dst:
    dst.write(downscaled, 1)

print(f"\nSaved: {out_tif}  ({h_c * UPSCALE} × {w_c * UPSCALE} px, 1/480°)")
print(f"Elapsed: {time.time() - _t0:.1f} s")

## Section 7 — Zonal Aggregation and Density Stratification

Computes per-zone statistics from the 1/480° downscaled raster using
`rasterstats` (paper §4.7). For each administrative unit, three quantities
are extracted: total downscaled CO₂ (tCO₂/yr), total resident population
(from GHSL GHS-POP at 100 m), and zone area (km²). Per-capita emissions
are derived (Eq. 8) and zones are stratified by population density
quartile, with a CO₂-OUTLIER flag for zones exceeding 5× the city-specific
median (paper §4.7).

In [ ]:
from rasterstats import zonal_stats

_t0 = time.time()

# --- 7.1 Zonal CO₂ statistics ---
print(f"Computing zonal CO₂ statistics on {len(gdf_zones)} units...")

stats_co2 = zonal_stats(
    gdf_zones,
    str(out_tif),
    stats=["sum", "mean", "std", "count"],
    nodata=0,
    all_touched=False,
)

zonal_df = gdf_zones[["_label"]].rename(columns={"_label": "zone"}).copy()
zonal_df["CO2_tco2_sum"]  = [round(s["sum"],  1) if s["sum"]  else 0.0 for s in stats_co2]
zonal_df["CO2_tco2_mean"] = [round(s["mean"], 2) if s["mean"] else 0.0 for s in stats_co2]
zonal_df["CO2_tco2_std"]  = [round(s["std"],  2) if s["std"]  else 0.0 for s in stats_co2]
zonal_df["n_pixels_fine"] = [s["count"] for s in stats_co2]

zonal_df = zonal_df[zonal_df["n_pixels_fine"] > 0].copy()
print(f"  Zones with valid pixels: {len(zonal_df)}")
print(f"  Coverage: {zonal_df['CO2_tco2_sum'].sum():,.0f} tCO₂/yr "
      f"({zonal_df['CO2_tco2_sum'].sum() / annual_tco2_1km.sum() * 100:.1f}% of input)")

# --- 7.2 Download GHSL population at 100 m for per-capita calculation ---
print(f"\nDownloading GHSL population at 100 m...")
pop_tif_100m = RESULT_DIR / "pop_count_100m.tif"
ghsl_pop_native = (ee.Image(f"JRC/GHSL/P2023A/GHS_POP/{ghsl_epoch_for_year(YEAR)}")
                    .select("population_count")
                    .clip(ROI))
geemap.ee_export_image(
    ghsl_pop_native, filename=str(pop_tif_100m),
    crs=EXPORT_CRS, scale=100, region=ROI, file_per_band=False,
)
with rasterio.open(pop_tif_100m) as src:
    pop_arr = src.read(1)
print(f"  Total population in AOI: {pop_arr[pop_arr > 0].sum():,.0f} inhabitants")

# --- 7.3 Zonal population statistics and per-capita merging ---
stats_pop = zonal_stats(
    gdf_zones, str(pop_tif_100m),
    stats=["sum", "count"], nodata=-9999,
)
pop_df = gdf_zones[["_label"]].rename(columns={"_label": "zone"}).copy()
pop_df["pop_tot"]      = [round(s["sum"])  if s["sum"]   and s["sum"] > 0 else 0
                          for s in stats_pop]
pop_df["n_pixels_pop"] = [s["count"]       if s["count"] else 0
                          for s in stats_pop]

merged = zonal_df.merge(pop_df, on="zone", how="left")
merged["CO2_per_capita"] = np.where(
    merged["pop_tot"] > 0,
    (merged["CO2_tco2_sum"] / merged["pop_tot"]).round(4),
    np.nan,
)

# Quality flag
def _quality(row):
    if row["pop_tot"] <= 0:        return "NO_POP"
    elif row["n_pixels_pop"] < 25: return "LOW"
    else:                          return "HIGH"

merged["pop_quality"] = merged.apply(_quality, axis=1)



# --- 7.4 Population density per zone (inhabitants per km²) ---
# Use UTM CRS for accurate area in km²
gdf_zones_utm = gdf_zones.to_crs(cfg["crs_utm"])
zone_area_km2 = gdf_zones_utm.geometry.area / 1e6
merged = merged.merge(
    pd.DataFrame({
        "zone":      gdf_zones["_label"].values,
        "area_km2":  zone_area_km2.values,
    }),
    on="zone", how="left",
)
merged["density_inh_km2"] = np.where(
    merged["area_km2"] > 0,
    (merged["pop_tot"] / merged["area_km2"]).round(1),
    np.nan,
)

# --- 7.5 City-relative stratification (paper §4.7) ---
# Step 1: flag CO₂-OUTLIER (per-capita > 5 × city median)
# Step 2: among remaining zones, partition by density quartile (Q25, Q75)
MIN_POP        = 500
MIN_POP_PIXELS = 25
OUTLIER_MULT   = 5.0

eligible = (
    (merged["pop_tot"]      >= MIN_POP) &
    (merged["n_pixels_pop"] >= MIN_POP_PIXELS) &
    merged["CO2_per_capita"].notna()
)

city_median_pc = merged.loc[eligible, "CO2_per_capita"].median()
outlier_thresh = OUTLIER_MULT * city_median_pc

# Density quartiles computed over NON-OUTLIER eligible zones
non_outlier = eligible & (merged["CO2_per_capita"] <= outlier_thresh)
Q25 = merged.loc[non_outlier, "density_inh_km2"].quantile(0.25)
Q75 = merged.loc[non_outlier, "density_inh_km2"].quantile(0.75)

def _classify(row):
    if not eligible[row.name]:
        return "EXCLUDED"
    if row["CO2_per_capita"] > outlier_thresh:
        return "CO2-OUTLIER"
    if row["density_inh_km2"] > Q75:
        return "DENSITY-HIGH"
    if row["density_inh_km2"] < Q25:
        return "DENSITY-LOW"
    return "DENSITY-MID"

merged["density_class"] = merged.apply(_classify, axis=1)

print(f"\nTable 5: City-relative stratification thresholds:")
print(f"  Median CO₂/capita:     {city_median_pc:.3f} tCO₂ cap⁻¹ yr⁻¹")
print(f"  CO₂-OUTLIER threshold: {outlier_thresh:.3f} tCO₂ cap⁻¹ yr⁻¹")
print(f"  Density Q25:           {Q25:,.0f} inh km⁻²")
print(f"  Density Q75:           {Q75:,.0f} inh km⁻²")
print(f"\nTable 6: Zone counts by class:")
print(merged["density_class"].value_counts().to_string())

# --- 7.6 Save zonal CSV and GeoPackage ---
csv_path = RESULT_DIR / f"zonal_statistics_{cfg['out_subdir']}.csv"
merged.to_csv(csv_path, index=False)
print(f"\nSaved: {csv_path}")

# -----------------------------------------------------------------------------
# 7.7 — Earth Observation profile for Table 5 (extended)
# -----------------------------------------------------------------------------
# Extends Table 5 with an Earth Observation morphological context block.
# Output is printed as a single continuous table (stratification thresholds
# from Section 7.5 + EO profile rows below) so the on-screen view already
# matches the paper Table 5 layout, and saved to a per-city CSV ready for
# cross-city consolidation.
# -----------------------------------------------------------------------------

from rasterio.features import geometry_mask

print(f"\nComputing Earth Observation profile for extended Table 5...")

# Union of valid (non-EXCLUDED) zones — same footprint used for the RF
valid_zones_gdf = gdf_zones.loc[
    gdf_zones["_label"].isin(merged.loc[merged["density_class"] != "EXCLUDED", "zone"])
].copy()
aoi_union_geom = unary_union(valid_zones_gdf.geometry)

with rasterio.open(feature_tif) as src:
    eo_transform = src.transform

# --- 7.7.1 Continuous-band zonal medians (then cross-zone median + IQR) -------
def _zonal_cross_stats(band_idx: int, nodata_value=0):
    """Zonal median per zone → cross-zone (median, Q25, Q75)."""
    band_arr = features_fine[band_idx].astype(np.float32)
    zs = zonal_stats(
        valid_zones_gdf, band_arr,
        affine=eo_transform, stats=["median"],
        nodata=nodata_value, all_touched=False,
    )
    vals = np.array(
        [s["median"] for s in zs if s["median"] is not None],
        dtype=np.float32,
    )
    if len(vals) == 0:
        return np.nan, np.nan, np.nan
    return float(np.median(vals)), float(np.quantile(vals, 0.25)), float(np.quantile(vals, 0.75))

built_med, built_q25, built_q75 = _zonal_cross_stats(BASE_BANDS.index("built_frac"))
bnr_med,   bnr_q25,   bnr_q75   = _zonal_cross_stats(BASE_BANDS.index("built_nres_frac"))
no2_med,   no2_q25,   no2_q75   = _zonal_cross_stats(BASE_BANDS.index("no2_trop"))
ndvi_med,  ndvi_q25,  ndvi_q75  = _zonal_cross_stats(BASE_BANDS.index("ndvi"))
elev_med,  elev_q25,  elev_q75  = _zonal_cross_stats(BASE_BANDS.index("elevation"), nodata_value=-32768)

# AOI-level scalars
elev_arr   = features_fine[BASE_BANDS.index("elevation")]
elev_valid = elev_arr[(elev_arr > -100) & np.isfinite(elev_arr)]
elev_range = float(elev_valid.max() - elev_valid.min())

# WorldCover: AOI-wide % coverage
aoi_fine_mask = geometry_mask(
    [aoi_union_geom.__geo_interface__],
    out_shape=(features_fine.shape[1], features_fine.shape[2]),
    transform=eo_transform,
    invert=True, all_touched=False,
)
def _wc_pct(band_name):
    idx = BASE_BANDS.index(band_name)
    return float(np.nanmean(features_fine[idx][aoi_fine_mask]) * 100.0)

wc_built_pct = _wc_pct("wc_built")
wc_tree_pct  = _wc_pct("wc_tree")
wc_crop_pct  = _wc_pct("wc_crop")
wc_water_pct = _wc_pct("wc_water")

# Total population & AOI-level density (from Section 7.3 outputs)
aoi_pop_total = merged.loc[merged["density_class"] != "EXCLUDED", "pop_tot"].sum()
aoi_area_km2  = merged.loc[merged["density_class"] != "EXCLUDED", "area_km2"].sum()
aoi_pop_density = aoi_pop_total / aoi_area_km2 if aoi_area_km2 > 0 else np.nan

# Convert NO2 from mol m⁻² to μmol m⁻² (× 1e6) for readable magnitudes
no2_med_umol = no2_med * 1e6
no2_q25_umol = no2_q25 * 1e6
no2_q75_umol = no2_q75 * 1e6

# --- 7.7.2 Assemble continuous Table 5 rows (thresholds + EO profile) ---------
def _fmt_iqr(med, q25, q75, decimals=3, unit_suffix=""):
    if pd.isna(med):
        return "—"
    if pd.isna(q25) or pd.isna(q75):
        return f"{med:.{decimals}f}{unit_suffix}"
    return f"{med:.{decimals}f} [{q25:.{decimals}f}–{q75:.{decimals}f}]{unit_suffix}"

def _fmt_int(x):
    return f"{x:,.0f}" if pd.notna(x) else "—"

# Each row: (parameter_label, value_formatted, raw_median, raw_q25, raw_q75, unit)
table5_rows = [
    # --- BLOCK A: Stratification thresholds (from Section 7.5) ---
    ("Valid administrative zones",
     f"{(merged['density_class'] != 'EXCLUDED').sum()}",
     (merged['density_class'] != 'EXCLUDED').sum(), np.nan, np.nan, "count"),
    ("Median CO₂/capita",
     f"{city_median_pc:.3f}",
     city_median_pc, np.nan, np.nan, "tCO₂ cap⁻¹ yr⁻¹"),
    ("CO₂-OUTLIER threshold (5 × median)",
     f"{outlier_thresh:.3f}",
     outlier_thresh, np.nan, np.nan, "tCO₂ cap⁻¹ yr⁻¹"),
    ("Population density Q25",
     _fmt_int(Q25),
     Q25, np.nan, np.nan, "inh km⁻²"),
    ("Population density Q75",
     _fmt_int(Q75),
     Q75, np.nan, np.nan, "inh km⁻²"),
    # --- BLOCK B: Earth Observation profile ---
    ("Total population (AOI)",
     _fmt_int(aoi_pop_total),
     aoi_pop_total, np.nan, np.nan, "inhabitants"),
    ("AOI population density",
     _fmt_int(aoi_pop_density),
     aoi_pop_density, np.nan, np.nan, "inh km⁻²"),
    ("Built-up fraction (GHSL)",
     _fmt_iqr(built_med, built_q25, built_q75, 3),
     built_med, built_q25, built_q75, "fraction"),
    ("Non-residential built fraction",
     _fmt_iqr(bnr_med, bnr_q25, bnr_q75, 3),
     bnr_med, bnr_q25, bnr_q75, "fraction"),
    ("Tropospheric NO₂ (TROPOMI)",
     _fmt_iqr(no2_med_umol, no2_q25_umol, no2_q75_umol, 1),
     no2_med_umol, no2_q25_umol, no2_q75_umol, "μmol m⁻²"),
    ("NDVI (Sentinel-2 annual median)",
     _fmt_iqr(ndvi_med, ndvi_q25, ndvi_q75, 3),
     ndvi_med, ndvi_q25, ndvi_q75, "index"),
    ("Elevation (NASADEM)",
     _fmt_iqr(elev_med, elev_q25, elev_q75, 1),
     elev_med, elev_q25, elev_q75, "m a.s.l."),
    ("Elevation range (max − min)",
     f"{elev_range:.1f}",
     elev_range, np.nan, np.nan, "m"),
    ("WorldCover Built-up coverage",
     f"{wc_built_pct:.1f}",
     wc_built_pct, np.nan, np.nan, "% AOI"),
    ("WorldCover Tree cover",
     f"{wc_tree_pct:.1f}",
     wc_tree_pct, np.nan, np.nan, "% AOI"),
    ("WorldCover Cropland",
     f"{wc_crop_pct:.1f}",
     wc_crop_pct, np.nan, np.nan, "% AOI"),
    ("WorldCover Water bodies",
     f"{wc_water_pct:.1f}",
     wc_water_pct, np.nan, np.nan, "% AOI"),
]

# --- 7.7.3 Save per-city CSV (raw numerics for cross-city merge) --------------
table5_df = pd.DataFrame(
    table5_rows,
    columns=["parameter", "value_formatted", "median", "q25", "q75", "unit"],
)
table5_df["city"]   = cfg["city_label"]
table5_df["subdir"] = cfg["out_subdir"]
table5_df = table5_df[
    ["city", "subdir", "parameter", "unit", "value_formatted", "median", "q25", "q75"]
]

eo_csv_path = RESULT_DIR / f"table5_full_{cfg['out_subdir']}.csv"
table5_df.to_csv(eo_csv_path, index=False)

# --- 7.7.4 On-screen print: single continuous Table 5 -------------------------
print(f"\nTable 5 (extended) — {cfg['city_label']}:")
print(f"{'Parameter':<40} {'Value':<32} {'Unit':<22}")
print("-" * 96)
# Visual separator between Block A (thresholds) and Block B (EO profile)
block_a_end = 5  # first 5 rows are thresholds
for i, row in enumerate(table5_rows):
    if i == block_a_end:
        print("-" * 96 + "   ← Earth Observation profile")
    print(f"{row[0]:<40} {row[1]:<32} {row[5]:<22}")

print(f"\nSaved: {eo_csv_path}")

# Geo-output with geometry
gdf_out = gdf_zones[["_label", "geometry"]].rename(columns={"_label": "zone"})
gdf_out = gdf_out.merge(
    merged.drop(columns="zone").assign(zone=merged["zone"]),
    on="zone", how="left",
)
gpkg_path = RESULT_DIR / f"zonal_statistics_{cfg['out_subdir']}.gpkg"
gdf_out.to_file(gpkg_path, driver="GPKG")
print(f"Saved: {gpkg_path}")
print(f"\nElapsed: {time.time() - _t0:.1f} s")

In [ ]:
# --- 7.6 Zonal CSV and GeoPackage (with EO predictor profile per zone) ---

# For every zone, compute the zonal-statistic appropriate to each band:
#   - Continuous bands (built_frac, built_nres_frac, pop_count, no2_trop,
#     ndvi, elevation): median + min + max — central tendency plus intra-zone
#     range (relevant for large heterogeneous administrative zones).
#   - Binary WorldCover bands (wc_built, wc_tree, wc_crop, wc_water): mean,
#     which equals the fraction of fine pixels classified as that class
#     within the zone. The median of a 0/1 raster collapses to 0 or 1 and
#     would discard the gradient information; min/max are trivially 0/1.
print(f"\nAdding EO predictor profile to zonal output ({len(BASE_BANDS)} bands)...")

EO_COLUMN_SUFFIXES = ["_median", "_min", "_max", "_frac", "_x", "_y"]
EO_BAND_NAMES = list(BASE_BANDS)  # 10 predictor bands

eo_cols_to_drop = [
    col for col in merged.columns
    if any(col == f"{b}{s}" for b in EO_BAND_NAMES for s in EO_COLUMN_SUFFIXES)
    or any(col.startswith(f"{b}_") and col.endswith(("_x", "_y")) for b in EO_BAND_NAMES)
]

if eo_cols_to_drop:
    print(f"Idempotency guard: removing {len(eo_cols_to_drop)} stale EO columns "
          f"from a previous run.")
    merged = merged.drop(columns=eo_cols_to_drop)

WORLDCOVER_BANDS = {"wc_built", "wc_tree", "wc_crop", "wc_water"}

with rasterio.open(feature_tif) as src:
    eo_transform = src.transform

for band_idx, band_name in enumerate(BASE_BANDS):
    nodata_val = -32768 if band_name == "elevation" else 0
    is_binary = band_name in WORLDCOVER_BANDS

    if is_binary:
        # Mean = coverage fraction (0–1). Skip min/max (always 0/1, uninformative).
        zs = zonal_stats(
            gdf_zones,
            features_fine[band_idx].astype(np.float32),
            affine=eo_transform,
            stats=["mean"],
            nodata=nodata_val,
            all_touched=False,
        )
        band_df = pd.DataFrame({
            "zone": gdf_zones["_label"].values,
            f"{band_name}_frac": [round(s["mean"], 4) if s["mean"] is not None else np.nan
                                  for s in zs],
        })
    else:
        # Continuous bands: median + min + max
        zs = zonal_stats(
            gdf_zones,
            features_fine[band_idx].astype(np.float32),
            affine=eo_transform,
            stats=["median", "min", "max"],
            nodata=nodata_val,
            all_touched=False,
        )
        band_df = pd.DataFrame({
            "zone": gdf_zones["_label"].values,
            f"{band_name}_median": [round(s["median"], 4) if s["median"] is not None else np.nan
                                    for s in zs],
            f"{band_name}_min":    [round(s["min"],    4) if s["min"]    is not None else np.nan
                                    for s in zs],
            f"{band_name}_max":    [round(s["max"],    4) if s["max"]    is not None else np.nan
                                    for s in zs],
        })

    merged = merged.merge(band_df, on="zone", how="left")

# Zonal sum del raster ODIAC sui poligoni amministrativi
stats_odiac_zones = zonal_stats(
    gdf_zones,
    str(odiac_annual_tif),
    stats=["sum"],
    nodata=0,
    all_touched=False,
)

odiac_total_in_admin = sum(
    s["sum"] for s in stats_odiac_zones if s["sum"] is not None
)

print(f"ODIAC totale ({cfg['city_label']}, anno {YEAR}):")
print(f"  Bbox clip (Sezione 1):           {annual_tco2_1km.sum():>15,.0f} tCO₂/yr")
print(f"  Dentro poligoni amministrativi:  {odiac_total_in_admin:>15,.0f} tCO₂/yr")
print(f"  Administrative units:            {len(gdf_zones)}")

# --- Save CSV ---
csv_path = RESULT_DIR / f"zonal_statistics_{cfg['out_subdir']}.csv"
merged.to_csv(csv_path, index=False)
print(f"Saved: {csv_path}")

# --- Save GeoPackage with geometry ---
gdf_out = gdf_zones[["_label", "geometry"]].rename(columns={"_label": "zone"})
gdf_out = gdf_out.merge(
    merged.drop(columns="zone").assign(zone=merged["zone"]),
    on="zone", how="left",
)
gpkg_path = RESULT_DIR / f"zonal_statistics_{cfg['out_subdir']}.gpkg"
gdf_out.to_file(gpkg_path, driver="GPKG")
print(f"Saved: {gpkg_path}")

print(f"\nElapsed: {time.time() - _t0:.1f} s")

## Section 8 — Per-Capita Emission Raster

Combines the 1/480° downscaled emission raster with a mass-conserving resample
of the GHS-POP population grid to obtain per-capita emissions at pixel level
(paper §4.7). Population is redistributed within each source cell so that the
zonal totals used in Section 7 are preserved.

In [ ]:
# ==============================================================================
# SECTION 8 — Per-Capita CO₂ Raster at 1/480° (mass-conserving population resample)
# ==============================================================================
#
# Combines the 1/480° downscaled CO₂ raster (Section 6) with GHSL GHS-POP
# at 100 m (Section 7) to produce a per-capita emission map at 1/480°.
#
# Critical methodological point: GHS-POP is a count (inhabitants per pixel),
# so the resampling to 1/480° MUST use sum-conserving aggregation, not
# bilinear or average. We use rasterio.warp.reproject with
# Resampling.sum, which integrates source-pixel counts into the larger
# target pixels. This guarantees that total population is preserved within
# the AOI to floating-point precision — the same mass-conservation logic
# applied to CO₂ in Section 6.
#
# The ratio CO2/pop is computed only where pop > MIN_POP_PER_PIXEL to avoid
# division by near-zero counts inflating per-capita values in unpopulated
# pixels (water, parks, industrial sites). Those pixels are set to NoData.
# ==============================================================================

from rasterio.warp import reproject, Resampling

_t0 = time.time()

# --- 8.1 Reproject GHS-POP from 100 m to the 1/480° CO₂ grid (sum-conserving) ---
print("Resampling GHS-POP (100 m) → 1/480° CO₂ grid using sum aggregation...")

with rasterio.open(pop_tif_100m) as src_pop:
    pop_src        = src_pop.read(1).astype(np.float32)
    pop_src        = np.where(np.isfinite(pop_src) & (pop_src > 0), pop_src, 0.0)
    src_transform  = src_pop.transform
    src_crs        = src_pop.crs

# Target grid: identical to the downscaled CO₂ raster
target_shape     = downscaled.shape                 # (h_c * UPSCALE, w_c * UPSCALE)
target_transform = transform_fine_clean             # from Section 6.6
target_crs       = "EPSG:4326"

pop_fine = np.zeros(target_shape, dtype=np.float32)

reproject(
    source=pop_src,
    destination=pop_fine,
    src_transform=src_transform,
    src_crs=src_crs,
    dst_transform=target_transform,
    dst_crs=target_crs,
    resampling=Resampling.sum,                      # sum-conserving: critical
)

# Apply the same AOI polygon mask used in Section 6.5
pop_fine = np.where(aoi_mask, pop_fine, 0.0).astype(np.float32)

# --- 8.2 Verify population conservation ---
pop_input_total  = float(pop_src[pop_src > 0].sum())
pop_output_total = float(pop_fine.sum())
pop_residual_pct = abs(pop_output_total - pop_input_total) / pop_input_total * 100

print(f"\nPopulation conservation check:")
print(f"  Input total  (100 m, GHSL):   {pop_input_total:>15,.0f} inhabitants")
print(f"  Output total (1/480°, summed):{pop_output_total:>15,.0f} inhabitants")
print(f"  Relative residual:            {pop_residual_pct:>15.4f}%")
# Small residual (< 1%) is expected because the AOI polygon mask trims a
# few border pixels that the source 100 m grid included partially.

# --- 8.3 Compute per-capita CO₂ at 1/480° ---
# Threshold: at least 1 inhabitant per fine pixel. Pixels below the threshold
# are not "no emissions per capita" — they have no resident population, so
# the ratio is undefined and we mask them as NoData.
MIN_POP_PER_PIXEL = 1.0

with np.errstate(divide="ignore", invalid="ignore"):
    per_capita = np.where(
        (pop_fine >= MIN_POP_PER_PIXEL) & aoi_mask,
        downscaled / pop_fine,                       # tCO₂/yr per inhabitant
        np.nan,
    ).astype(np.float32)

# --- 8.4 Diagnostics ---
valid_pc = per_capita[np.isfinite(per_capita)]
print(f"\nPer-capita CO₂ raster (1/480°):")
print(f"  Valid pixels (pop ≥ {MIN_POP_PER_PIXEL:.0f}): {len(valid_pc):,} / {aoi_mask.sum():,} "
      f"({len(valid_pc) / aoi_mask.sum() * 100:.1f}% of AOI)")
print(f"  Min:    {valid_pc.min():>10.3f} tCO₂ cap⁻¹ yr⁻¹")
print(f"  P25:    {np.percentile(valid_pc, 25):>10.3f}")
print(f"  Median: {np.median(valid_pc):>10.3f}")
print(f"  P75:    {np.percentile(valid_pc, 75):>10.3f}")
print(f"  P99:    {np.percentile(valid_pc, 99):>10.3f}")
print(f"  Max:    {valid_pc.max():>10.3f}")

# Sanity check: pixel-level median should be roughly comparable in order of
# magnitude to the zone-level median from Section 7 (0.571 for Milan).
# It will not match exactly because pixel-level ratios are unweighted while
# zone-level ratios are population-weighted within each zone.

# --- 8.5 Save the per-capita GeoTIFF ---
profile_pc = {
    "driver":    "GTiff",
    "dtype":     "float32",
    "width":     target_shape[1],
    "height":    target_shape[0],
    "count":     1,
    "crs":       target_crs,
    "transform": target_transform,
    "nodata":    -9999.0,
    "compress":  "lzw",
}
per_capita_out = np.where(np.isfinite(per_capita), per_capita, -9999.0).astype(np.float32)

out_pc_tif  = RESULT_DIR / f"per_capita_CO2_{cfg['out_subdir']}_1-480deg.tif"
out_pop_tif = RESULT_DIR / f"pop_count_{cfg['out_subdir']}_1-480deg.tif"

with rasterio.open(out_pc_tif, "w", **profile_pc) as dst:
    dst.write(per_capita_out, 1)

# Also save the sum-resampled population raster (useful diagnostic + reusable)
profile_pop_fine = {**profile_pc, "nodata": 0.0}
with rasterio.open(out_pop_tif, "w", **profile_pop_fine) as dst:
    dst.write(pop_fine, 1)

print(f"\nSaved: {out_pc_tif}")
print(f"Saved: {out_pop_tif}")

# --- 8.6 Quick visualisation (log scale, robust to outliers) ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Panel 1: CO₂ downscaled
co2_plot = np.where(downscaled > 0, downscaled, np.nan)
im0 = axes[0].imshow(co2_plot, cmap="YlOrRd",
                     vmin=0, vmax=np.nanpercentile(co2_plot, 99))
axes[0].set_title(f"Downscaled CO₂ (tCO₂/yr/pixel)\n{cfg['city_label']}, 1/480°")
plt.colorbar(im0, ax=axes[0], fraction=0.046)

# Panel 2: Population
pop_plot = np.where(pop_fine > 0, pop_fine, np.nan)
im1 = axes[1].imshow(pop_plot, cmap="Blues",
                     vmin=0, vmax=np.nanpercentile(pop_plot, 99))
axes[1].set_title(f"Population (inh/pixel)\nGHSL → 1/480°, sum-resampled")
plt.colorbar(im1, ax=axes[1], fraction=0.046)

# Panel 3: Per-capita CO₂ (clipped at p99 to avoid outlier-driven colormap)
pc_plot   = per_capita.copy()
pc_vmax   = np.nanpercentile(valid_pc, 99)
im2 = axes[2].imshow(pc_plot, cmap="viridis", vmin=0, vmax=pc_vmax)
axes[2].set_title(f"Per-capita CO₂ (tCO₂ cap⁻¹ yr⁻¹)\nclipped at p99 = {pc_vmax:.2f}")
plt.colorbar(im2, ax=axes[2], fraction=0.046)

for ax in axes:
    ax.set_xticks([]); ax.set_yticks([])

plt.tight_layout()
plt.savefig(RESULT_DIR / f"per_capita_CO2_{cfg['out_subdir']}_panels.png",
            dpi=150, bbox_inches="tight")
plt.show()

print(f"\nElapsed: {time.time() - _t0:.1f} s")

## Section 9 — Uncertainty Quantification via Spatial Block Bootstrap

Estimates how stable the learned redistribution is (paper §4.8, §5.5). The
spatial CV blocks are resampled with replacement; for each replicate the model
is refitted and the share assigned to every fine pixel is recomputed. The
dispersion of those shares gives a pixel-level coefficient of variation, and
the replicate-wise zonal totals give intervals for the per-class statistics.

The procedure characterises the stability of the redistribution conditional on
the ODIAC input; it does not quantify the absolute accuracy of the underlying
inventory.

In [ ]:
# =============================================================================
# SECTION 9 — Uncertainty Quantification via Spatial Block Bootstrap
# =============================================================================
# Quantifies the predictive uncertainty of the mass-preserving redistribution
# (paper §6.3). Three design decisions define what is being measured:
#
# 1. UNCERTAINTY IS COMPUTED ON THE REDISTRIBUTION SHARES, NOT ON RAW RF
#    SCORES. Under the mass-conservation constraint the output of each fine
#    pixel is w = s_ij / SUM_P(s_kl), so errors within a parent cell are
#    negatively correlated by construction. Dispersion measured on raw scores
#    systematically overstates the uncertainty of the delivered product.
#
# 2. THE RESAMPLING UNIT IS THE SPATIAL BLOCK, NOT THE TREE. Between-tree
#    dispersion in a Random Forest reflects deliberate decorrelation of base
#    learners (max_features='sqrt'), not predictive uncertainty of the
#    ensemble. Resampling the 8x8 km blocks with replacement and refitting
#    the full forest measures the uncertainty of the ensemble itself, and is
#    consistent with the Spatial Block GroupKFold design of paper §4.5.
#
# 3. EACH TREE IS BACK-TRANSFORMED BEFORE NORMALISATION. The model is fitted
#    in log1p space; expm1(mean) != mean(expm1), so back-transformation must
#    precede aggregation.
#
# SCOPE. The resulting intervals are CONDITIONAL ON ODIAC. They propagate
# neither the magnitude uncertainty of the inventory nor the uncertainty of
# its point-source allocation, both of which dominate in absolute terms
# (Gurney et al., 2019). This must be stated explicitly in the manuscript.
# =============================================================================

from sklearn.ensemble import RandomForestRegressor
from sklearn.utils import resample
from rasterstats import zonal_stats

_t0 = time.time()

# --- 9.0 Configuration -------------------------------------------------------
UQ_CONFIG = {
    "n_replicates":  50,        # bootstrap replicates (50 is adequate for 5-95 CI)
    "ci_low":        5,         # lower percentile of reported interval
    "ci_high":       95,        # upper percentile of reported interval
    "seed_offset":   1000,      # keeps replicate seeds disjoint from SHARED seed
    "min_pop":       500,       # same eligibility threshold as paper §4.7
}

H_F, W_F = h_c * UPSCALE, w_c * UPSCALE
N_FINE   = UPSCALE * UPSCALE
UNIFORM  = 1.0 / N_FINE
B        = UQ_CONFIG["n_replicates"]

# Parent ODIAC totals broadcast to the fine grid (rebuilt to avoid depending
# on intermediate variables from Section 6).
odiac_parent_up = np.repeat(np.repeat(annual_tco2_1km, UPSCALE, axis=0),
                            UPSCALE, axis=1)

zone_labels = gdf_zones["_label"].values
n_zones     = len(zone_labels)

print(f"Spatial block bootstrap — {cfg['city_label']}")
print(f"  Replicates:        {B}")
print(f"  Resampling unit:   {len(np.unique(groups_valid))} spatial blocks "
      f"({SPATIAL_BLOCK_SIZE}x{SPATIAL_BLOCK_SIZE} coarse cells)")
print(f"  Fine grid:         {H_F} x {W_F} px, {aoi_mask.sum():,} inside AOI")
print()


def _shares_from_scores(score_flat: np.ndarray) -> np.ndarray:
    """
    Convert a flat vector of fine-grid RF scores into mass-preserving
    redistribution shares.

    Each parent cell contributes exactly UPSCALE^2 fine pixels whose shares
    sum to one. Degenerate parents (all scores zero, e.g. open water with no
    built-up or population signal) fall back to uniform allocation, matching
    the behaviour of Section 6.3.
    """
    s = score_flat.astype(np.float32).reshape(H_F, W_F)
    s_parent = s.reshape(h_c, UPSCALE, w_c, UPSCALE).sum(axis=(1, 3))
    s_parent_up = np.repeat(np.repeat(s_parent, UPSCALE, axis=0), UPSCALE, axis=1)
    return np.where(s_parent_up > 0, s / s_parent_up, UNIFORM).astype(np.float32)


# --- 9.1 Bootstrap loop ------------------------------------------------------
blocks = np.unique(groups_valid)

sum_w   = np.zeros((H_F, W_F), dtype=np.float64)   # accumulator for share mean
sum_w2  = np.zeros((H_F, W_F), dtype=np.float64)   # accumulator for share variance
Z       = np.full((B, n_zones), np.nan, dtype=np.float64)   # zonal CO2 per replicate
n_failed = 0

for b in range(B):
    try:
        sel_blocks = resample(blocks, replace=True,
                              random_state=UQ_CONFIG["seed_offset"] + b)
        idx = np.concatenate([np.where(groups_valid == g)[0] for g in sel_blocks])

        model_b = RandomForestRegressor(
            **best_params,
            random_state=UQ_CONFIG["seed_offset"] + b,
            n_jobs=-1,
        )
        model_b.fit(X_valid[idx], np.log1p(y_valid[idx]))

        score_b = np.expm1(model_b.predict(X_fine)).clip(min=0)
        w_b     = _shares_from_scores(score_b)

        sum_w  += w_b
        sum_w2 += w_b.astype(np.float64) ** 2

        # Redistributed emissions for this replicate, masked to the AOI polygon
        co2_b = np.where(aoi_mask, w_b * odiac_parent_up, 0.0).astype(np.float32)

        zs_b = zonal_stats(gdf_zones, co2_b, affine=transform_fine_clean,
                           stats=["sum"], nodata=0, all_touched=False)
        Z[b, :] = [s["sum"] if s["sum"] is not None else 0.0 for s in zs_b]

    except Exception as exc:
        n_failed += 1
        print(f"  Replicate {b} failed ({type(exc).__name__}): {exc}")
        continue

    if (b + 1) % 10 == 0:
        print(f"  {b + 1}/{B} replicates")

n_ok = B - n_failed
if n_ok < 10:
    raise RuntimeError(f"Only {n_ok} replicates completed; results unusable.")
if n_failed:
    print(f"  WARNING: {n_failed} replicate(s) failed and were skipped.")

# --- 9.2 Pixel-level share dispersion ----------------------------------------
mean_w = sum_w / n_ok
var_w  = (sum_w2 - n_ok * mean_w ** 2) / (n_ok - 1)
std_w  = np.sqrt(np.clip(var_w, 0, None))

with np.errstate(divide="ignore", invalid="ignore"):
    cv_w = np.where((mean_w > 0) & aoi_mask, std_w / mean_w, np.nan).astype(np.float32)

# Diagnostic 1: degenerate parents, where CV is identically zero by
# construction rather than by model agreement. These must be excluded from
# any claim about spatial reliability.
score_ref  = np.expm1(final_model.predict(X_fine)).clip(min=0).reshape(H_F, W_F)
parent_sum = score_ref.reshape(h_c, UPSCALE, w_c, UPSCALE).sum(axis=(1, 3))
degenerate = np.repeat(np.repeat(parent_sum == 0, UPSCALE, 0), UPSCALE, 1) & aoi_mask

# Diagnostic 2: if CV is strongly anti-correlated with the mean share, it is
# largely reproducing the emission map itself and carries little independent
# information; the absolute standard deviation should be reported instead.
_m = np.isfinite(cv_w) & (mean_w > 0) & aoi_mask & ~degenerate
corr_cv_share = float(np.corrcoef(np.log(cv_w[_m]), np.log(mean_w[_m]))[0, 1])

cv_tif = RESULT_DIR / f"uq_share_cv_{cfg['out_subdir']}_1-480deg.tif"
profile_cv = {
    "driver": "GTiff", "dtype": "float32",
    "width": W_F, "height": H_F, "count": 1,
    "crs": "EPSG:4326", "transform": transform_fine_clean,
    "nodata": -9999.0, "compress": "lzw",
}
with rasterio.open(cv_tif, "w", **profile_cv) as dst:
    dst.write(np.where(np.isfinite(cv_w), cv_w, -9999.0).astype(np.float32), 1)
    dst.set_band_description(1, "Bootstrap CV of redistribution shares (dimensionless)")

print(f"\nPixel-level share dispersion:")
print(f"  Median CV over AOI:              {np.nanmedian(cv_w[~degenerate]):.3f}")
print(f"  Degenerate parents (CV artefact): {degenerate.sum():,} px "
      f"({degenerate.sum() / aoi_mask.sum() * 100:.1f}% of AOI)")
print(f"  Corr(log CV, log mean share):     {corr_cv_share:+.3f}")
if corr_cv_share < -0.70:
    print("  NOTE: CV largely tracks the emission map; report absolute sigma instead.")

# --- 9.3 Zonal intervals -----------------------------------------------------
Z_ok = Z[~np.isnan(Z).all(axis=1)]

uq_zonal = pd.DataFrame({
    "zone":            zone_labels,
    "CO2_boot_median": np.nanmedian(Z_ok, axis=0),
    "CO2_boot_lo":     np.nanpercentile(Z_ok, UQ_CONFIG["ci_low"],  axis=0),
    "CO2_boot_hi":     np.nanpercentile(Z_ok, UQ_CONFIG["ci_high"], axis=0),
    "CO2_boot_cv":     np.nanstd(Z_ok, axis=0, ddof=1) / np.nanmean(Z_ok, axis=0),
})

uq_zonal = uq_zonal.merge(
    merged[["zone", "pop_tot", "density_class", "CO2_per_capita"]],
    on="zone", how="left",
)

for src_col, dst_col in [("CO2_boot_median", "pc_boot_median"),
                         ("CO2_boot_lo",     "pc_boot_lo"),
                         ("CO2_boot_hi",     "pc_boot_hi")]:
    uq_zonal[dst_col] = np.where(uq_zonal["pop_tot"] > 0,
                                 uq_zonal[src_col] / uq_zonal["pop_tot"],
                                 np.nan)

uq_csv = RESULT_DIR / f"uq_zonal_{cfg['out_subdir']}.csv"
uq_zonal.round(4).to_csv(uq_csv, index=False)

# --- 9.4 Within-class per-capita distributions across replicates -------------
# For every replicate, the class-level per-capita median is recomputed from
# scratch. Storing the full replicate vector (rather than a summary) is what
# allows the cross-city ratios of Table 6 to be given intervals in §9.5.
pop_vec   = uq_zonal["pop_tot"].values.astype(float)
class_vec = uq_zonal["density_class"].values

eligible_zone = (pop_vec >= UQ_CONFIG["min_pop"]) & np.isfinite(pop_vec) & (pop_vec > 0)

with np.errstate(divide="ignore", invalid="ignore"):
    PC = np.where(pop_vec[None, :] > 0, Z_ok / pop_vec[None, :], np.nan)

REPORT_CLASSES = ["DENSITY-HIGH", "DENSITY-MID", "DENSITY-LOW", "CO2-OUTLIER"]
class_reps = {}
for cls in REPORT_CLASSES:
    sel = eligible_zone & (class_vec == cls)
    class_reps[cls] = (np.nanmedian(PC[:, sel], axis=1) if sel.sum() > 0
                       else np.full(len(PC), np.nan))

npz_path = RESULT_DIR / f"uq_class_replicates_{cfg['out_subdir']}.npz"
np.savez(
    npz_path,
    city=cfg["city_label"],
    subdir=cfg["out_subdir"],
    n_replicates=n_ok,
    classes=np.array(REPORT_CLASSES),
    replicates=np.vstack([class_reps[c] for c in REPORT_CLASSES]),  # (n_class, B)
    zone_counts=np.array([int((eligible_zone & (class_vec == c)).sum())
                          for c in REPORT_CLASSES]),
)

print(f"\nWithin-class per-capita (tCO2 cap-1 yr-1), "
      f"{UQ_CONFIG['ci_low']}-{UQ_CONFIG['ci_high']} percentile interval:")
print(f"{'Class':<16} {'n':>3} {'median':>9} {'interval':>22}")
print("-" * 54)
for cls in REPORT_CLASSES:
    r = class_reps[cls]
    n = int((eligible_zone & (class_vec == cls)).sum())
    if n == 0 or np.all(np.isnan(r)):
        print(f"{cls:<16} {n:>3} {'--':>9} {'--':>22}")
        continue
    lo, hi = np.nanpercentile(r, [UQ_CONFIG["ci_low"], UQ_CONFIG["ci_high"]])
    print(f"{cls:<16} {n:>3} {np.nanmedian(r):>9.3f} "
          f"{'[' + f'{lo:.3f}-{hi:.3f}' + ']':>22}")

# --- 9.5 Cross-city ratio intervals (runs once both cities are available) ----
# Replicates are drawn independently in each city, so replicate i of Shanghai
# is paired with replicate i of Milan arbitrarily. The pairing is valid only
# because replicates are i.i.d. within each city; this must be stated in the
# methods section.
other_subdir = "milan" if cfg["out_subdir"] != "milan" else "shanghai_pudong"
other_npz    = Path("results") / other_subdir / f"uq_class_replicates_{other_subdir}.npz"

if other_npz.exists():
    other = np.load(other_npz, allow_pickle=True)
    sh_key = "shanghai_pudong"
    this_is_sh = cfg["out_subdir"] == sh_key

    reps_this  = np.vstack([class_reps[c] for c in REPORT_CLASSES])
    reps_other = other["replicates"]
    n_pair     = min(reps_this.shape[1], reps_other.shape[1])

    num = reps_this[:, :n_pair] if this_is_sh else reps_other[:, :n_pair]
    den = reps_other[:, :n_pair] if this_is_sh else reps_this[:, :n_pair]

    print(f"\nCross-city per-capita ratio Shanghai / Milan "
          f"({n_pair} paired replicates):")
    print(f"{'Class':<16} {'ratio':>9} {'interval':>22}")
    print("-" * 50)
    for i, cls in enumerate(REPORT_CLASSES):
        with np.errstate(divide="ignore", invalid="ignore"):
            ratio = np.where(den[i] > 0, num[i] / den[i], np.nan)
        if np.all(np.isnan(ratio)):
            print(f"{cls:<16} {'--':>9} {'--':>22}")
            continue
        lo, hi = np.nanpercentile(ratio, [UQ_CONFIG["ci_low"], UQ_CONFIG["ci_high"]])
        print(f"{cls:<16} {np.nanmedian(ratio):>8.2f}x "
              f"{'[' + f'{lo:.2f}-{hi:.2f}' + ']':>22}")
    print("\n  Non-overlapping intervals across classes indicate that the "
          "\n  departure from a single aggregate ratio is structural (paper §6.1.1).")
else:
    print(f"\nCross-city ratios pending: run the notebook with "
          f"CITY = '{other_subdir}' to generate {other_npz.name}.")

# --- 9.6 Summary -------------------------------------------------------------
print(f"\nSaved: {cv_tif.name}")
print(f"Saved: {uq_csv.name}")
print(f"Saved: {npz_path.name}")
print(f"Elapsed: {time.time() - _t0:.1f} s")

## Section 10 — Summary and Verification

The first cell prints the metrics reported in the manuscript for the selected
city. The second compares the values produced by this run against the published
ones and reports any divergence field by field, so that a deviation is visible
immediately rather than being discovered downstream.

In [ ]:
print("=" * 70)
print(f"  PIPELINE COMPLETE — {cfg['city_label']}, reference year {YEAR}")
print("=" * 70)
print()
print("Model performance (paper §5.1, Table 3):")
print(f"  CV R² (Spatial Block GroupKFold, original space): {cv_r2_final:.3f}")
print(f"  Train R²:                                         {r2_train_full:.3f}")
print(f"  Train/CV gap:                                     {gap_original:.3f}")
print(f"  Selected hyperparameters:                         {best_params}")
print()
print("Mass conservation (paper §4.6, Eq. 7):")
print(f"  ODIAC input total:     {mass_input:>15,.0f} tCO₂/yr")
print(f"  Downscaled total:      {mass_output:>15,.0f} tCO₂/yr")
print(f"  Relative residual:     {delta_mass_pct:>15.6f}%")
print()
print("Cross-city per-capita comparison (paper §5.4, Tables 5–6):")
print(f"  Median CO₂/capita:     {city_median_pc:.3f} tCO₂ cap⁻¹ yr⁻¹")
print(f"  CO₂-OUTLIER threshold: {outlier_thresh:.3f} tCO₂ cap⁻¹ yr⁻¹")
print(f"  Density Q25 / Q75:     {Q25:,.0f} / {Q75:,.0f} inh km⁻²")
print()
print("Output files in:", RESULT_DIR.resolve())
for f in sorted(RESULT_DIR.glob("*")):
    if f.is_file():
        print(f"  {f.name}  ({f.stat().st_size / 1024:.0f} KB)")

In [ ]:
# =============================================================================
# VERIFICATION — compare this run against the values reported in the manuscript
# =============================================================================
# Reference values are those of the published runs (post bbox-clip pipeline).
# Comparison is field by field with explicit tolerances. A divergence does not
# stop execution; it is reported so that the cause can be traced before the
# outputs are used or deposited.
# =============================================================================

REFERENCE = {
    "shanghai_pudong": {
        "grid_dims":        (85, 102),
        "n_blocks":         143,
        "odiac_total":      67_981_928,
        "cv_r2":            0.650,
        "train_r2":         0.744,
        "gap":              0.094,
        "max_depth":        10,
        "min_samples_leaf": 10,
        "n_estimators":     200,
        "median_pc":        1.560,
        "outlier_thresh":   7.802,
        "Q25":              3_898,
        "Q75":              27_465,
        "class_counts":     {"DENSITY-HIGH": 9, "DENSITY-MID": 16, "DENSITY-LOW": 9,
                             "CO2-OUTLIER": 1, "EXCLUDED": 1},
    },
    "milan": {
        "grid_dims":        (48, 105),
        "n_blocks":         84,
        "odiac_total":      4_778_557,
        "cv_r2":            0.666,
        "train_r2":         0.761,
        "gap":              0.096,
        "max_depth":        10,
        "min_samples_leaf": 15,
        "n_estimators":     200,
        "median_pc":        0.571,
        "outlier_thresh":   2.857,
        "Q25":              982,
        "Q75":              3_494,
        "class_counts":     {"DENSITY-HIGH": 34, "DENSITY-MID": 68, "DENSITY-LOW": 34,
                             "CO2-OUTLIER": 4, "EXCLUDED": 0},
    },
}

ref   = REFERENCE[CITY]
rows  = []


def _check(label, obtained, expected, tol=0.0, rel=False, fmt="{:,.3f}"):
    """Record one comparison. tol=0 means exact match."""
    if isinstance(expected, (int, float)) and isinstance(obtained, (int, float)):
        diff = abs(obtained - expected)
        limit = abs(expected) * tol if rel else tol
        ok = diff <= limit
        rows.append((label, fmt.format(obtained), fmt.format(expected), "OK" if ok else "MISMATCH"))
    else:
        ok = obtained == expected
        rows.append((label, str(obtained), str(expected), "OK" if ok else "MISMATCH"))
    return ok


# --- Grid geometry and training sample ---------------------------------------
_check("Coarse grid (rows × cols)", tuple(sorted((h_c, w_c))), tuple(sorted(ref["grid_dims"])), fmt="{}")
_check("Coarse grid cells", h_c * w_c, ref["grid_dims"][0] * ref["grid_dims"][1], fmt="{:,.0f}")
_check("Spatial CV blocks", int(np.unique(groups_valid).size), ref["n_blocks"], fmt="{:,.0f}")

# --- Inventory ----------------------------------------------------------------
_check("ODIAC bbox total (tCO₂/yr)", float(annual_tco2_1km.sum()), ref["odiac_total"],
       tol=0.001, rel=True, fmt="{:,.0f}")

# --- Model --------------------------------------------------------------------
_check("CV R² (original space)", cv_r2_final,   ref["cv_r2"],    tol=0.005)
_check("Train R² (original space)", r2_train_full, ref["train_r2"], tol=0.005)
_check("Train/CV gap",           gap_original,  ref["gap"],      tol=0.005)
_check("max_depth",        best_params["max_depth"],        ref["max_depth"], fmt="{}")
_check("min_samples_leaf", best_params["min_samples_leaf"], ref["min_samples_leaf"], fmt="{}")
_check("n_estimators",     best_params["n_estimators"],     ref["n_estimators"], fmt="{}")

# --- Mass conservation --------------------------------------------------------
_check("Mass residual (%)", abs(delta_mass_pct), 0.0, tol=1e-4, fmt="{:.6f}")

# --- Zonal stratification -----------------------------------------------------
_check("Median CO₂/capita", city_median_pc,  ref["median_pc"],      tol=0.005, rel=True)
_check("CO₂-OUTLIER threshold", outlier_thresh, ref["outlier_thresh"], tol=0.005, rel=True)
_check("Density Q25 (inh/km²)", float(Q25), float(ref["Q25"]), tol=0.01, rel=True, fmt="{:,.0f}")
_check("Density Q75 (inh/km²)", float(Q75), float(ref["Q75"]), tol=0.01, rel=True, fmt="{:,.0f}")

_counts = merged["density_class"].value_counts().to_dict()
for _cls, _n in ref["class_counts"].items():
    _check(f"Zones — {_cls}", int(_counts.get(_cls, 0)), _n, fmt="{}")

# --- Report -------------------------------------------------------------------
verification = pd.DataFrame(rows, columns=["check", "this_run", "manuscript", "status"])
ver_csv = RESULT_DIR / f"verification_{cfg['out_subdir']}.csv"
verification.to_csv(ver_csv, index=False)

_w = max(len(r[0]) for r in rows) + 2
print("=" * (_w + 44))
print(f"  VERIFICATION AGAINST MANUSCRIPT — {cfg['city_label']}")
print("=" * (_w + 44))
print(f"  {'check':<{_w}}{'this run':>16}{'manuscript':>16}   status")
print("-" * (_w + 44))
for label, got, exp, status in rows:
    flag = "" if status == "OK" else "   <<<"
    print(f"  {label:<{_w}}{got:>16}{exp:>16}   {status}{flag}")
print("-" * (_w + 44))

n_bad = int((verification["status"] != "OK").sum())
VERIFICATION_PASSED = n_bad == 0

if VERIFICATION_PASSED:
    print(f"  All {len(rows)} checks passed — outputs match the manuscript.")
else:
    print(f"  {n_bad} of {len(rows)} checks diverge from the manuscript.")
    print("  Inspect the flagged rows before using or depositing these outputs.")
    print("  Most frequent causes: a different ODIAC vintage, a modified study-area")
    print("  polygon, or an Earth Engine predictor composite built on other dates.")
print("=" * (_w + 44))
print(f"\nSaved: {ver_csv}")

### Packaging the outputs

Collects everything written to the results folder into a single archive for
download and deposit.

In [ ]:
# --- Zip & download the entire `results/` folder in one go (Colab) ---
import shutil
from pathlib import Path

# Identify the results folder for this run (same RESULT_DIR used throughout the notebook)
results_root = Path("results")  # parent of shanghai_pudong/ or milan/
zip_basename = f"results_{cfg['out_subdir']}_{pd.Timestamp.now():%Y%m%d_%H%M}"
zip_path = Path(f"/content/{zip_basename}.zip")

# Create the zip (shutil.make_archive expects the path WITHOUT the .zip extension)
archive_path = shutil.make_archive(
    base_name=str(zip_path.with_suffix("")),  # /content/results_xxx_YYYYMMDD_HHMM
    format="zip",
    root_dir=str(results_root.parent),         # /content
    base_dir=str(results_root.name),           # 'results' — relative, preserves folder structure
)

zip_size_mb = Path(archive_path).stat().st_size / (1024 * 1024)
print(f"Created: {archive_path}")
print(f"Size:    {zip_size_mb:.2f} MB")

# Trigger browser download (Colab only)
try:
    from google.colab import files  # type: ignore
    files.download(archive_path)
    print(f"Download triggered.")
except (ImportError, ModuleNotFoundError):
    print("Not running in Colab — zip is available at the path above.")